# **Project Name**    -   Paisabazaar Banking Fraud Analysis



##### **Project Type**    - Classification

# **Project Summary -**

Credit Score Prediction Using Machine Learning and Feature Importance Analysis

This project focuses on developing a machine learning model to predict the credit score category of customers using their financial and credit-related information. The dataset, containing 100,000 customer records and 28 attributes, includes information such as annual income, monthly salary, credit utilization, outstanding debt, payment behavior, loan details, and credit history.

A comprehensive data preprocessing pipeline was implemented to prepare the dataset for machine learning. This involved converting categorical variables into numerical representations using techniques such as One-Hot Encoding and MultiLabelBinarizer, handling ambiguous values in the Payment_of_Min_Amount feature through a Decision Tree-based prediction approach, and encoding the target variable using Label Encoding.

Two supervised learning algorithms, Decision Tree Classifier and Random Forest Classifier, were evaluated for credit score prediction. Hyperparameter tuning was performed using manual experimentation and GridSearchCV for the Decision Tree model. The Random Forest Classifier significantly outperformed the Decision Tree model, achieving an accuracy of approximately 82.1%.

To improve model interpretability and reduce feature dimensionality, permutation feature importance was applied iteratively. Features with low importance were removed over multiple iterations until a compact set of 18 highly influential features was obtained. The reduced-feature model maintained nearly the same predictive performance (approximately 81.9% accuracy), demonstrating that a smaller subset of financial attributes is sufficient for effective credit score prediction.

The project concludes that ensemble learning combined with feature importance analysis can produce an accurate, computationally efficient, and interpretable credit scoring model. Future work includes integrating explainable AI techniques such as SHAP or LIME to provide transparent explanations for individual predictions.

# **GitHub Link -**

Provide your GitHub Link here.

# **Problem Statement**


Financial institutions rely on credit scores to evaluate a customer's creditworthiness before approving loans, credit cards, or other financial services. Traditional credit assessment methods often involve analyzing numerous customer attributes, many of which may contribute little to the final prediction. High-dimensional datasets can increase computational complexity, reduce model interpretability, and potentially introduce redundant information.

The objective of this project is to develop a machine learning model capable of accurately predicting customer credit score categories using financial and credit-related information. In addition to achieving high predictive performance, the project aims to identify the most influential features affecting credit score prediction through feature importance analysis, thereby reducing model complexity without significantly sacrificing accuracy.

Specifically, the project addresses the following objectives:

Develop a robust preprocessing pipeline for handling categorical variables and missing or ambiguous values.
Compare the predictive performance of Decision Tree and Random Forest classifiers.
Optimize model performance through hyperparameter tuning.
Identify and eliminate less significant features using permutation feature importance.
Build a compact and efficient prediction model that maintains high classification accuracy while improving interpretability.

# ***Let's Begin !***

## ***1. Know Your Data***

https://www.kaggle.com/datasets/utkarshrawat19/paisabazaar-banking-fraud-dataset/data

### Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier

from sklearn.inspection import permutation_importance

import joblib

# Set pandas to display all columns
pd.set_option('display.max_columns', None)

### Dataset Loading

In [ ]:
# Mounting google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Load Dataset
data = pd.read_csv("/content/drive/MyDrive/Paisabazaar Banking Fraud Dataset.csv")

### Dataset First View

In [ ]:
# Dataset First Look
data.head()

,ID,Customer_ID,Month,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Type_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,Num_Credit_Inquiries,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score
0,5634,3392,1,Aaron Maashoh,23.0,821000265.0,Scientist,19114.12,1824.843333,3.0,4.0,3.0,4.0,"Auto Loan, Credit-Builder Loan, Personal Loan,...",3.0,7.0,11.27,4.0,Good,809.98,26.822620,265.0,No,49.574949,21.46538,High_spent_Small_value_payments,312.494089,Good
1,5635,3392,2,Aaron Maashoh,23.0,821000265.0,Scientist,19114.12,1824.843333,3.0,4.0,3.0,4.0,"Auto Loan, Credit-Builder Loan, Personal Loan,...",3.0,4.0,11.27,4.0,Good,809.98,31.944960,266.0,No,49.574949,21.46538,Low_spent_Large_value_payments,284.629162,Good
2,5636,3392,3,Aaron Maashoh,23.0,821000265.0,Scientist,19114.12,1824.843333,3.0,4.0,3.0,4.0,"Auto Loan, Credit-Builder Loan, Personal Loan,...",3.0,7.0,11.27,4.0,Good,809.98,28.609352,267.0,No,49.574949,21.46538,Low_spent_Medium_value_payments,331.209863,Good
3,5637,3392,4,Aaron Maashoh,23.0,821000265.0,Scientist,19114.12,1824.843333,3.0,4.0,3.0,4.0,"Auto Loan, Credit-Builder Loan, Personal Loan,...",5.0,4.0,6.27,4.0,Good,809.98,31.377862,268.0,No,49.574949,21.46538,Low_spent_Small_value_payments,223.451310,Good
4,5638,3392,5,Aaron Maashoh,23.0,821000265.0,Scientist,19114.12,1824.843333,3.0,4.0,3.0,4.0,"Auto Loan, Credit-Builder Loan, Personal Loan,...",6.0,4.0,11.27,4.0,Good,809.98,24.797347,269.0,No,49.574949,21.46538,High_spent_Medium_value_payments,341.489231,Good


### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
data.shape

(100000, 28)

### Dataset Information

In [ ]:
# Dataset Info
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 28 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   ID                        100000 non-null  int64  
 1   Customer_ID               100000 non-null  int64  
 2   Month                     100000 non-null  int64  
 3   Name                      100000 non-null  object 
 4   Age                       100000 non-null  float64
 5   SSN                       100000 non-null  float64
 6   Occupation                100000 non-null  object 
 7   Annual_Income             100000 non-null  float64
 8   Monthly_Inhand_Salary     100000 non-null  float64
 9   Num_Bank_Accounts         100000 non-null  float64
 10  Num_Credit_Card           100000 non-null  float64
 11  Interest_Rate             100000 non-null  float64
 12  Num_of_Loan               100000 non-null  float64
 13  Type_of_Loan              100000 non-null  ob

#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
data.duplicated().sum()

np.int64(0)

#### Missing Values/Null Values

In [ ]:
# Missing Values
data.isna().sum()

,0
ID,0
Customer_ID,0
Month,0
Name,0
Age,0
SSN,0
Occupation,0
Annual_Income,0
Monthly_Inhand_Salary,0
Num_Bank_Accounts,0


In [ ]:
data.isnull().sum()

,0
ID,0
Customer_ID,0
Month,0
Name,0
Age,0
SSN,0
Occupation,0
Annual_Income,0
Monthly_Inhand_Salary,0
Num_Bank_Accounts,0


## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns
data.columns

Index(['ID', 'Customer_ID', 'Month', 'Name', 'Age', 'SSN', 'Occupation',
       'Annual_Income', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts',
       'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan', 'Type_of_Loan',
       'Delay_from_due_date', 'Num_of_Delayed_Payment', 'Changed_Credit_Limit',
       'Num_Credit_Inquiries', 'Credit_Mix', 'Outstanding_Debt',
       'Credit_Utilization_Ratio', 'Credit_History_Age',
       'Payment_of_Min_Amount', 'Total_EMI_per_month',
       'Amount_invested_monthly', 'Payment_Behaviour', 'Monthly_Balance',
       'Credit_Score'],
      dtype='object')

In [ ]:
# Dataset Describe
data.describe()

,ID,Customer_ID,Month,Age,SSN,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,Num_Credit_Inquiries,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Total_EMI_per_month,Amount_invested_monthly,Monthly_Balance
count,100000.000000,100000.000000,100000.000000,100000.000000,1.000000e+05,100000.000000,100000.000000,100000.000000,100000.000000,100000.00000,100000.000000,100000.00000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000
mean,80631.500000,25982.666640,4.500000,33.316340,5.004617e+08,50505.123449,4197.270835,5.368820,5.533570,14.53208,3.532880,21.08141,13.313120,10.470323,5.798250,1426.220376,32.285173,221.220460,107.699208,55.101315,392.697586
std,43301.486619,14340.543051,2.291299,10.764812,2.908267e+08,38299.422093,3186.432497,2.593314,2.067098,8.74133,2.446356,14.80456,6.237166,6.609481,3.867826,1155.129026,5.116875,99.680716,132.267056,39.006932,201.652719
min,5634.000000,1006.000000,1.000000,14.000000,8.134900e+04,7005.930000,303.645417,0.000000,0.000000,1.00000,0.000000,0.00000,0.000000,0.500000,0.000000,0.230000,20.000000,1.000000,0.000000,0.000000,0.007760
25%,43132.750000,13664.500000,2.750000,24.000000,2.451686e+08,19342.972500,1626.594167,3.000000,4.000000,7.00000,2.000000,10.00000,9.000000,5.380000,3.000000,566.072500,28.052567,144.000000,29.268886,27.959111,267.615983
50%,80631.500000,25777.000000,4.500000,33.000000,5.006886e+08,36999.705000,3095.905000,5.000000,5.000000,13.00000,3.000000,18.00000,14.000000,9.400000,5.000000,1166.155000,32.305784,219.000000,66.462304,45.156550,333.865366
75%,118130.250000,38385.000000,6.250000,42.000000,7.560027e+08,71683.470000,5957.715000,7.000000,7.000000,20.00000,5.000000,28.00000,18.000000,14.850000,8.000000,1945.962500,36.496663,302.000000,147.392573,71.295797,463.215683
max,155629.000000,50999.000000,8.000000,56.000000,9.999934e+08,179987.280000,15204.633333,11.000000,11.000000,34.00000,9.000000,62.00000,25.000000,29.980000,17.000000,4998.070000,50.000000,404.000000,1779.103254,434.191089,1183.930696


### Check Unique Values for each variable.

#### Month

In [ ]:
data["Month"].value_counts()

,count
Month,
1,12500
2,12500
3,12500
4,12500
5,12500
6,12500
7,12500
8,12500


In [ ]:
data["Month"].unique()

array([1, 2, 3, 4, 5, 6, 7, 8])

#### Age

In [ ]:
data["Age"].value_counts()

,count
Age,
38.0,3070
28.0,3045
31.0,3037
26.0,3025
32.0,2969
36.0,2953
25.0,2952
27.0,2951
35.0,2940


In [ ]:
data["Age"].unique()

array([23., 28., 34., 54., 55., 21., 31., 33., 30., 24., 44., 45., 40.,
       41., 32., 35., 36., 39., 37., 20., 46., 26., 42., 19., 48., 38.,
       43., 22., 16., 18., 15., 27., 25., 14., 17., 47., 53., 56., 29.,
       49., 51., 50., 52.])

#### Occupation

In [ ]:
data["Occupation"].value_counts()

,count
Occupation,
Lawyer,7096
Engineer,6864
Architect,6824
Mechanic,6776
Accountant,6744
Scientist,6744
Media_Manager,6720
Developer,6720
Teacher,6672


In [ ]:
data["Occupation"].unique()

array(['Scientist', 'Teacher', 'Engineer', 'Entrepreneur', 'Developer',
       'Lawyer', 'Media_Manager', 'Doctor', 'Journalist', 'Manager',
       'Accountant', 'Musician', 'Mechanic', 'Writer', 'Architect'],
      dtype=object)

#### Num_Bank_Accounts

In [ ]:
data["Num_Bank_Accounts"].unique()

array([ 3.,  2.,  1.,  7.,  4.,  0.,  8.,  5.,  6.,  9., 10., 11.])

In [ ]:
data["Num_Bank_Accounts"].value_counts()

,count
Num_Bank_Accounts,
6.0,13175
7.0,12999
8.0,12940
4.0,12343
5.0,12298
3.0,12107
9.0,5503
10.0,5329
1.0,4540


#### Num_Credit_Card

In [ ]:
data["Num_Credit_Card"].unique()

array([ 4.,  5.,  1.,  7.,  6.,  8.,  3.,  9.,  2., 10., 11.,  0.])

In [ ]:
data["Num_Credit_Card"].value_counts()

,count
Num_Credit_Card,
5.0,18903
7.0,17024
6.0,16932
4.0,14362
3.0,13560
8.0,5073
10.0,4962
9.0,4753
2.0,2196


#### Interest_Rate

An interest rate is expressed as a percentage of the principal - the original amount of money borrowed or deposited - and indicates the cost of borrowing or the return on savings over a specific period, usually a year.

For borrowers, it represents the extra amount they must pay in addition to the principal.

For lenders or savers, it is the compensation for allowing others to use their money.

In [ ]:
data["Interest_Rate"].unique()

array([ 3.,  6.,  8.,  4.,  5., 15.,  7., 12., 20.,  1., 14., 32., 16.,
       17., 10., 31., 25., 18., 19.,  9., 24., 13., 33., 11., 21., 29.,
       28., 30., 23., 34.,  2., 27., 26., 22.])

In [ ]:
data["Interest_Rate"].value_counts()

,count
Interest_Rate,
8.0,5104
5.0,5096
6.0,4832
12.0,4648
10.0,4616
7.0,4584
9.0,4576
11.0,4512
18.0,4192


#### Num_of_Loan

In [ ]:
data["Num_of_Loan"].unique()

array([4., 1., 3., 0., 2., 7., 5., 6., 8., 9.])

In [ ]:
data["Num_of_Loan"].value_counts()

,count
Num_of_Loan,
3.0,15752
2.0,15712
4.0,15456
0.0,11408
1.0,11128
6.0,8144
7.0,7680
5.0,7528
9.0,3856


#### Type_of_Loan

In [ ]:
len(data["Type_of_Loan"].unique())

6261

In [ ]:
data["Type_of_Loan"].value_counts()

,count
Type_of_Loan,
No Data,11408
Not Specified,1408
Credit-Builder Loan,1280
Personal Loan,1272
Debt Consolidation Loan,1264
...,...
"Student Loan, Auto Loan, Student Loan, Credit-Builder Loan, Home Equity Loan, Debt Consolidation Loan, and Debt Consolidation Loan",8
"Debt Consolidation Loan, Personal Loan, Mortgage Loan, Personal Loan, Not Specified, Mortgage Loan, and Home Equity Loan",8
"Student Loan, Home Equity Loan, Student Loan, Personal Loan, Not Specified, Auto Loan, Auto Loan, and Debt Consolidation Loan",8


In [ ]:
data["Type_of_Loan"].head()

,Type_of_Loan
0,"Auto Loan, Credit-Builder Loan, Personal Loan,..."
1,"Auto Loan, Credit-Builder Loan, Personal Loan,..."
2,"Auto Loan, Credit-Builder Loan, Personal Loan,..."
3,"Auto Loan, Credit-Builder Loan, Personal Loan,..."
4,"Auto Loan, Credit-Builder Loan, Personal Loan,..."


Here, we need to use MultiLabelBinarizer concept on "Type_of_Loan" column.

#### Delay_from_due_date

In [ ]:
data["Delay_from_due_date"].unique()

array([ 3.,  5.,  6.,  8.,  7., 13., 10.,  0.,  4.,  9.,  1., 12., 11.,
       30., 31., 34., 27., 14.,  2., 16., 17., 15., 23., 22., 21., 18.,
       19., 52., 51., 48., 53., 26., 43., 28., 25., 20., 47., 46., 49.,
       24., 61., 29., 50., 58., 45., 59., 55., 56., 57., 54., 62., 36.,
       41., 33., 32., 39., 44., 42., 60., 35., 38., 40., 37.])

In [ ]:
data["Delay_from_due_date"].value_counts()

,count
Delay_from_due_date,
15.0,3596
13.0,3424
8.0,3324
14.0,3313
10.0,3281
...,...
59.0,528
39.0,525
43.0,502


#### Num_of_Delayed_Payment

In [ ]:
data["Num_of_Delayed_Payment"].unique()

array([ 7.,  4.,  8.,  6.,  1.,  3.,  0.,  5.,  9., 15., 12., 17., 10.,
        2., 11., 14., 20., 22., 13., 16., 19., 18., 21., 23., 24., 25.])

In [ ]:
data["Num_of_Delayed_Payment"].value_counts()

,count
Num_of_Delayed_Payment,
19.0,5982
17.0,5832
10.0,5802
16.0,5768
15.0,5724
18.0,5668
20.0,5584
12.0,5493
9.0,5399


#### Num_Credit_Inquiries

In [ ]:
data["Num_Credit_Inquiries"].unique()

array([ 4.,  2.,  3.,  5.,  9.,  8.,  7.,  6.,  0.,  1., 10., 11., 12.,
       17., 13., 14., 16., 15.])

In [ ]:
data["Num_Credit_Inquiries"].value_counts()

,count
Num_Credit_Inquiries,
4.0,11690
3.0,9188
6.0,8399
7.0,8362
2.0,8335
8.0,8133
1.0,7796
0.0,7190
5.0,5951


#### Credit_Mix

In [ ]:
data["Credit_Mix"].unique()

array(['Good', 'Standard', 'Bad'], dtype=object)

In [ ]:
data["Credit_Mix"].value_counts()

,count
Credit_Mix,
Standard,45848
Good,30384
Bad,23768


#### Payment_of_Min_Amount

In [ ]:
data["Payment_of_Min_Amount"].unique()

array(['No', 'NM', 'Yes'], dtype=object)

In [ ]:
data["Payment_of_Min_Amount"].value_counts()

,count
Payment_of_Min_Amount,
Yes,52326
No,35667
NM,12007


#### Payment_Behaviour

In [ ]:
data["Payment_Behaviour"].unique()

array(['High_spent_Small_value_payments',
       'Low_spent_Large_value_payments',
       'Low_spent_Medium_value_payments',
       'Low_spent_Small_value_payments',
       'High_spent_Medium_value_payments',
       'High_spent_Large_value_payments'], dtype=object)

In [ ]:
data["Payment_Behaviour"].value_counts()

,count
Payment_Behaviour,
Low_spent_Small_value_payments,28616
High_spent_Medium_value_payments,19738
High_spent_Large_value_payments,14726
Low_spent_Medium_value_payments,14399
High_spent_Small_value_payments,11764
Low_spent_Large_value_payments,10757


#### Credit_Score

In [ ]:
data["Credit_Score"].unique()

array(['Good', 'Standard', 'Poor'], dtype=object)

In [ ]:
data["Credit_Score"].value_counts()

,count
Credit_Score,
Standard,53174
Poor,28998
Good,17828


## 3. ***Data Wrangling***

### 1. Changing datatype from float to int

In [ ]:
# These columns should be in integer format only
float_cols = ["Age", "Num_Bank_Accounts", "Num_Credit_Card", "Num_of_Loan", "Delay_from_due_date", "Num_of_Delayed_Payment", "Num_Credit_Inquiries", "Credit_History_Age"]

for col in float_cols:
  data[col] = data[col].astype(int)

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 28 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   ID                        100000 non-null  int64  
 1   Customer_ID               100000 non-null  int64  
 2   Month                     100000 non-null  int64  
 3   Name                      100000 non-null  object 
 4   Age                       100000 non-null  int64  
 5   SSN                       100000 non-null  float64
 6   Occupation                100000 non-null  object 
 7   Annual_Income             100000 non-null  float64
 8   Monthly_Inhand_Salary     100000 non-null  float64
 9   Num_Bank_Accounts         100000 non-null  int64  
 10  Num_Credit_Card           100000 non-null  int64  
 11  Interest_Rate             100000 non-null  float64
 12  Num_of_Loan               100000 non-null  int64  
 13  Type_of_Loan              100000 non-null  ob

In [ ]:
data["Age"].head()

,Age
0,23
1,23
2,23
3,23
4,23


### 2. Saving Original Dataset

In [ ]:
data_original = data

### 3. Mapping month numbers to month names

In [ ]:
data["Month"].unique()

array([1, 2, 3, 4, 5, 6, 7, 8])

In [ ]:
month_map = {
    1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr',
    5: 'May', 6: 'Jun', 7: 'Jul', 8: 'Aug',
}

In [ ]:
data["Month_Names"] = data["Month"].map(month_map)

data["Month_Names"].head()

,Month_Names
0,Jan
1,Feb
2,Mar
3,Apr
4,May


In [ ]:
data["Month_Names"].isna().sum()

np.int64(0)

In [ ]:
data["Month_Names"].isnull().sum()

np.int64(0)

### 3. Deleting uneccessary columns : ID, Customer_ID, Name, SSN

In [ ]:
data = data.drop(["ID", "Customer_ID", "Name", "SSN"], axis = 1)

data.head()

,Month,Age,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Type_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,Num_Credit_Inquiries,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score,Month_Names
0,1,23,Scientist,19114.12,1824.843333,3,4,3.0,4,"Auto Loan, Credit-Builder Loan, Personal Loan,...",3,7,11.27,4,Good,809.98,26.822620,265,No,49.574949,21.46538,High_spent_Small_value_payments,312.494089,Good,Jan
1,2,23,Scientist,19114.12,1824.843333,3,4,3.0,4,"Auto Loan, Credit-Builder Loan, Personal Loan,...",3,4,11.27,4,Good,809.98,31.944960,266,No,49.574949,21.46538,Low_spent_Large_value_payments,284.629162,Good,Feb
2,3,23,Scientist,19114.12,1824.843333,3,4,3.0,4,"Auto Loan, Credit-Builder Loan, Personal Loan,...",3,7,11.27,4,Good,809.98,28.609352,267,No,49.574949,21.46538,Low_spent_Medium_value_payments,331.209863,Good,Mar
3,4,23,Scientist,19114.12,1824.843333,3,4,3.0,4,"Auto Loan, Credit-Builder Loan, Personal Loan,...",5,4,6.27,4,Good,809.98,31.377862,268,No,49.574949,21.46538,Low_spent_Small_value_payments,223.451310,Good,Apr
4,5,23,Scientist,19114.12,1824.843333,3,4,3.0,4,"Auto Loan, Credit-Builder Loan, Personal Loan,...",6,4,11.27,4,Good,809.98,24.797347,269,No,49.574949,21.46538,High_spent_Medium_value_payments,341.489231,Good,May


In [ ]:
data.shape

(100000, 25)

In [ ]:
data.columns

Index(['Month', 'Age', 'Occupation', 'Annual_Income', 'Monthly_Inhand_Salary',
       'Num_Bank_Accounts', 'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan',
       'Type_of_Loan', 'Delay_from_due_date', 'Num_of_Delayed_Payment',
       'Changed_Credit_Limit', 'Num_Credit_Inquiries', 'Credit_Mix',
       'Outstanding_Debt', 'Credit_Utilization_Ratio', 'Credit_History_Age',
       'Payment_of_Min_Amount', 'Total_EMI_per_month',
       'Amount_invested_monthly', 'Payment_Behaviour', 'Monthly_Balance',
       'Credit_Score', 'Month_Names'],
      dtype='object')

### 4. Num_Bank_Accounts == 0

This credit score data might be based on past financial activity. That could be the reason behind the large number of customers are having zero number of bank accounts.

In [ ]:
data[data["Num_Bank_Accounts"] == 0].head()

,Month,Age,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Type_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,Num_Credit_Inquiries,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score,Month_Names
48,1,33,Lawyer,131313.4,11242.783333,0,1,8.0,2,"Credit-Builder Loan, and Mortgage Loan",0,3,9.34,2,Good,352.16,32.200509,367,NM,137.644605,86.566388,High_spent_Medium_value_payments,858.462474,Good,Jan
49,2,34,Lawyer,131313.4,11242.783333,0,1,8.0,2,"Credit-Builder Loan, and Mortgage Loan",0,2,15.34,4,Good,352.16,31.983710,368,No,137.644605,86.566388,High_spent_Small_value_payments,547.760457,Good,Feb
50,3,34,Lawyer,131313.4,10469.207759,0,1,8.0,2,"Credit-Builder Loan, and Mortgage Loan",0,3,9.34,4,Good,352.16,31.803134,369,NM,911.220179,86.566388,High_spent_Large_value_payments,1038.569407,Good,Mar
51,4,34,Lawyer,131313.4,10469.207759,0,1,8.0,2,"Credit-Builder Loan, and Mortgage Loan",0,2,8.34,4,Good,352.16,42.645785,370,No,911.220179,86.566388,High_spent_Medium_value_payments,899.198772,Good,Apr
52,5,34,Lawyer,131313.4,10469.207759,0,1,8.0,2,"Credit-Builder Loan, and Mortgage Loan",0,4,9.34,4,Good,352.16,40.902517,371,No,911.220179,86.566388,High_spent_Large_value_payments,963.254819,Good,May


In [ ]:
data["Num_Bank_Accounts"].head()

,Num_Bank_Accounts
0,3
1,3
2,3
3,3
4,3


In [ ]:
data["Num_Bank_Accounts"].value_counts()

,count
Num_Bank_Accounts,
6,13175
7,12999
8,12940
4,12343
5,12298
3,12107
9,5503
10,5329
1,4540


In [ ]:
data[data["Num_Bank_Accounts"] == 0].shape

(4417, 25)

Building model with and without bank accounts = 0

In [ ]:
data_with_ba_zero = data

In [ ]:
data_with_ba_zero.shape

(100000, 25)

In [ ]:
data_without_ba_zero = data[data["Num_Bank_Accounts"] > 0]

In [ ]:
data_without_ba_zero.shape

(95583, 25)

There are only 9 customers with 11 bank accounts. Keeping these entries for ML algorithm to learn from it too.

### 5. Num_Credit_Card == 0

Some people build credit through loans only or can have a credit score without even applying for a credit card.

In [ ]:
data[data["Num_Credit_Card"] == 0].shape

(14, 25)

In [ ]:
data[data["Num_Credit_Card"] == 0]

,Month,Age,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Type_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,Num_Credit_Inquiries,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score,Month_Names
4716,5,24,Writer,101912.13,8503.677500,4,0,4.0,0,No Data,5,2,4.83,2,Good,152.60,36.707990,243,No,0.000000,66.232053,Low_spent_Large_value_payments,773.432634,Good,May
4717,6,24,Writer,101912.13,8503.677500,4,0,4.0,0,No Data,5,6,4.83,2,Good,152.60,36.928316,244,No,0.000000,66.232053,High_spent_Large_value_payments,1024.135697,Standard,Jun
4718,7,24,Writer,101912.13,8503.677500,4,0,4.0,0,No Data,5,6,4.83,2,Good,152.60,26.630283,245,NM,0.000000,66.232053,Low_spent_Medium_value_payments,842.938582,Good,Jul
4719,8,24,Writer,101912.13,8503.677500,4,0,4.0,0,No Data,5,3,4.83,2,Good,152.60,42.515861,246,No,0.000000,66.232053,Low_spent_Medium_value_payments,719.094083,Good,Aug
9278,7,23,Mechanic,116721.18,9552.765000,1,0,12.0,1,Payday Loan,4,8,10.12,3,Good,48.88,41.498049,273,NM,82.914222,94.261036,High_spent_Small_value_payments,722.132496,Standard,Jul
9279,8,23,Mechanic,116721.18,9552.765000,1,0,12.0,1,Payday Loan,4,9,10.12,3,Good,48.88,33.270882,274,No,82.914222,94.261036,High_spent_Medium_value_payments,880.415397,Good,Aug
10588,5,29,Musician,79548.32,6355.026667,4,0,3.0,4,"Not Specified, Debt Consolidation Loan, Home E...",7,4,3.30,3,Good,870.51,25.285875,360,No,211.214525,103.417571,High_spent_Medium_value_payments,446.088480,Good,May
10589,6,29,Musician,79548.32,6355.026667,4,0,3.0,4,"Not Specified, Debt Consolidation Loan, Home E...",7,4,8.30,3,Good,870.51,33.656790,361,No,211.214525,103.417571,High_spent_Large_value_payments,535.461220,Good,Jun
10590,7,29,Musician,79548.32,6355.026667,4,0,3.0,4,"Not Specified, Debt Consolidation Loan, Home E...",11,6,3.30,3,Good,870.51,30.690501,362,No,211.214525,103.417571,Low_spent_Medium_value_payments,153.081724,Good,Jul
10591,8,29,Musician,79548.32,6355.026667,4,0,3.0,4,"Not Specified, Debt Consolidation Loan, Home E...",7,4,3.30,3,Good,870.51,35.619724,363,No,211.214525,103.417571,Low_spent_Large_value_payments,123.905655,Good,Aug


### 6. Num_of_Loan == 0

A customer might have taken loans in the past, repaid them completely, and currently have zero loans.

They might have credit cards which contribute to their credit history, and not loans.

In [ ]:
no_loan = data[data["Num_of_Loan"] == 0]

In [ ]:
no_loan.head(10)

,Month,Age,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Type_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,Num_Credit_Inquiries,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score,Month_Names
32,1,21,Developer,35547.71,2853.309167,7,5,5.0,0,No Data,5,15,2.58,4,Standard,943.86,39.797764,368,Yes,0.0,37.643638,High_spent_Medium_value_payments,288.605522,Standard,Jan
33,2,21,Developer,35547.71,2853.309167,7,5,5.0,0,No Data,9,15,2.58,4,Standard,943.86,27.020360,369,NM,0.0,37.643638,High_spent_Medium_value_payments,460.887276,Standard,Feb
34,3,21,Developer,35547.71,2853.309167,7,5,5.0,0,No Data,5,12,2.58,4,Standard,943.86,23.462303,370,Yes,0.0,37.643638,Low_spent_Medium_value_payments,392.192266,Standard,Mar
35,4,21,Developer,35547.71,2853.309167,7,5,5.0,0,No Data,1,15,2.58,4,Standard,943.86,28.924954,371,Yes,0.0,37.643638,High_spent_Medium_value_payments,438.545432,Standard,Apr
36,5,21,Developer,35547.71,2853.309167,7,5,5.0,0,No Data,9,17,2.58,4,Standard,943.86,41.776187,372,Yes,0.0,37.643638,High_spent_Small_value_payments,482.607638,Standard,May
37,6,21,Developer,35547.71,2853.309167,7,5,5.0,0,No Data,5,15,2.58,4,Standard,943.86,29.217556,373,Yes,0.0,37.643638,High_spent_Medium_value_payments,497.687279,Standard,Jun
38,7,21,Developer,35547.71,2853.309167,7,5,5.0,0,No Data,10,15,2.58,4,Standard,943.86,26.263823,374,Yes,0.0,37.643638,Low_spent_Small_value_payments,394.318934,Standard,Jul
39,8,21,Developer,35547.71,2853.309167,7,5,5.0,0,No Data,1,15,2.58,4,Standard,943.86,25.862922,375,Yes,0.0,37.643638,High_spent_Small_value_payments,364.000016,Standard,Aug
40,1,31,Lawyer,73928.46,5988.705000,4,5,8.0,0,No Data,12,10,10.14,2,Good,548.20,39.962685,384,No,0.0,42.635590,High_spent_Large_value_payments,740.196090,Good,Jan
41,2,31,Lawyer,73928.46,5988.705000,4,5,8.0,0,No Data,8,7,10.14,2,Good,548.20,42.769864,384,NM,0.0,42.635590,Low_spent_Medium_value_payments,705.931286,Good,Feb


In [ ]:
data["Num_of_Loan"].value_counts()

,count
Num_of_Loan,
3,15752
2,15712
4,15456
0,11408
1,11128
6,8144
7,7680
5,7528
9,3856


In [ ]:
no_loan["Num_Credit_Card"].describe()

,Num_Credit_Card
count,11408.000000
mean,4.685659
std,1.660096
min,0.000000
25%,3.000000
50%,5.000000
75%,6.000000
max,8.000000


In [ ]:
data.columns

Index(['Month', 'Age', 'Occupation', 'Annual_Income', 'Monthly_Inhand_Salary',
       'Num_Bank_Accounts', 'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan',
       'Type_of_Loan', 'Delay_from_due_date', 'Num_of_Delayed_Payment',
       'Changed_Credit_Limit', 'Num_Credit_Inquiries', 'Credit_Mix',
       'Outstanding_Debt', 'Credit_Utilization_Ratio', 'Credit_History_Age',
       'Payment_of_Min_Amount', 'Total_EMI_per_month',
       'Amount_invested_monthly', 'Payment_Behaviour', 'Monthly_Balance',
       'Credit_Score', 'Month_Names'],
      dtype='object')

### 7. Monthly_Inhand_Salary * 12 <= Annual_Income

There are many other factors which contribute to annual income and not only the monthly inhand salary.

In [ ]:
data[(data["Monthly_Inhand_Salary"] * 12) > data["Annual_Income"]].shape

(48754, 25)

### 8. Type_of_Loan : MultiLabelBinarizer

#### Working of pandas explode() function

* DataFrame.explode(column, ignore_index=False)

* Transform each element of a list-like to a row, replicating index values.

In [ ]:
first_row = data["Type_of_Loan"][0].split(",")

second_row = data["Type_of_Loan"][0].split(",")

print(first_row)

print(second_row)

['Auto Loan', ' Credit-Builder Loan', ' Personal Loan', ' and Home Equity Loan']
['Auto Loan', ' Credit-Builder Loan', ' Personal Loan', ' and Home Equity Loan']


In [ ]:
df = pd.DataFrame({"Person_1" : [first_row], "Person_2" : [second_row]})

df

,Person_1,Person_2
0,"[Auto Loan, Credit-Builder Loan, Personal Lo...","[Auto Loan, Credit-Builder Loan, Personal Lo..."


In [ ]:
df["Person_1"].explode().str.strip()

,Person_1
0,Auto Loan
0,Credit-Builder Loan
0,Personal Loan
0,and Home Equity Loan


#### How many unique loan types are there?

In [ ]:
loan_types = data["Type_of_Loan"].fillna("").str.split(",").explode().str.strip()

loan_types.head(10)

,Type_of_Loan
0,Auto Loan
0,Credit-Builder Loan
0,Personal Loan
0,and Home Equity Loan
1,Auto Loan
1,Credit-Builder Loan
1,Personal Loan
1,and Home Equity Loan
2,Auto Loan
2,Credit-Builder Loan


In [ ]:
loan_types.value_counts()

,count
Type_of_Loan,
Credit-Builder Loan,31920
Payday Loan,31664
Not Specified,31008
Mortgage Loan,30504
Home Equity Loan,30440
Personal Loan,30224
Student Loan,30176
Debt Consolidation Loan,30072
Auto Loan,29816


In [ ]:
len(loan_types.unique())

19

Total 19 loan types are there.

#### Converting rows of "Type_of_Loan" column into list --> each row will be converted into list

In [ ]:
loan_lists = data['Type_of_Loan'].fillna('').apply(lambda x: [i.strip() for i in x.split(',') if i.strip()])

loan_lists.head(10)

,Type_of_Loan
0,"[Auto Loan, Credit-Builder Loan, Personal Loan..."
1,"[Auto Loan, Credit-Builder Loan, Personal Loan..."
2,"[Auto Loan, Credit-Builder Loan, Personal Loan..."
3,"[Auto Loan, Credit-Builder Loan, Personal Loan..."
4,"[Auto Loan, Credit-Builder Loan, Personal Loan..."
5,"[Auto Loan, Credit-Builder Loan, Personal Loan..."
6,"[Auto Loan, Credit-Builder Loan, Personal Loan..."
7,"[Auto Loan, Credit-Builder Loan, Personal Loan..."
8,[Credit-Builder Loan]
9,[Credit-Builder Loan]


#### Starting with MultiLabelBinarizer

In [ ]:
mlb = MultiLabelBinarizer()

In [ ]:
loan_encoded = pd.DataFrame(mlb.fit_transform(loan_lists), columns=mlb.classes_, index=data.index)

In [ ]:
loan_encoded.head(10)

,Auto Loan,Credit-Builder Loan,Debt Consolidation Loan,Home Equity Loan,Mortgage Loan,No Data,Not Specified,Payday Loan,Personal Loan,Student Loan,and Auto Loan,and Credit-Builder Loan,and Debt Consolidation Loan,and Home Equity Loan,and Mortgage Loan,and Not Specified,and Payday Loan,and Personal Loan,and Student Loan
0,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0
1,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0
2,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0
3,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0
4,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0
5,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0
6,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0
7,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0
8,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
9,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [ ]:
data_encoded = pd.concat([data.drop('Type_of_Loan', axis=1), loan_encoded], axis=1)

In [ ]:
data_encoded.head(5)

,Month,Age,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,Num_Credit_Inquiries,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score,Month_Names,Auto Loan,Credit-Builder Loan,Debt Consolidation Loan,Home Equity Loan,Mortgage Loan,No Data,Not Specified,Payday Loan,Personal Loan,Student Loan,and Auto Loan,and Credit-Builder Loan,and Debt Consolidation Loan,and Home Equity Loan,and Mortgage Loan,and Not Specified,and Payday Loan,and Personal Loan,and Student Loan
0,1,23,Scientist,19114.12,1824.843333,3,4,3.0,4,3,7,11.27,4,Good,809.98,26.822620,265,No,49.574949,21.46538,High_spent_Small_value_payments,312.494089,Good,Jan,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0
1,2,23,Scientist,19114.12,1824.843333,3,4,3.0,4,3,4,11.27,4,Good,809.98,31.944960,266,No,49.574949,21.46538,Low_spent_Large_value_payments,284.629162,Good,Feb,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0
2,3,23,Scientist,19114.12,1824.843333,3,4,3.0,4,3,7,11.27,4,Good,809.98,28.609352,267,No,49.574949,21.46538,Low_spent_Medium_value_payments,331.209863,Good,Mar,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0
3,4,23,Scientist,19114.12,1824.843333,3,4,3.0,4,5,4,6.27,4,Good,809.98,31.377862,268,No,49.574949,21.46538,Low_spent_Small_value_payments,223.451310,Good,Apr,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0
4,5,23,Scientist,19114.12,1824.843333,3,4,3.0,4,6,4,11.27,4,Good,809.98,24.797347,269,No,49.574949,21.46538,High_spent_Medium_value_payments,341.489231,Good,May,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0


In [ ]:
data_encoded.shape

(100000, 43)

### 9. Further Encoding on Features except Payment_of_Min_Amount

In [ ]:
data_encoded.head()

,Month,Age,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,Num_Credit_Inquiries,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score,Month_Names,Auto Loan,Credit-Builder Loan,Debt Consolidation Loan,Home Equity Loan,Mortgage Loan,No Data,Not Specified,Payday Loan,Personal Loan,Student Loan,and Auto Loan,and Credit-Builder Loan,and Debt Consolidation Loan,and Home Equity Loan,and Mortgage Loan,and Not Specified,and Payday Loan,and Personal Loan,and Student Loan
0,1,23,Scientist,19114.12,1824.843333,3,4,3.0,4,3,7,11.27,4,Good,809.98,26.822620,265,No,49.574949,21.46538,High_spent_Small_value_payments,312.494089,Good,Jan,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0
1,2,23,Scientist,19114.12,1824.843333,3,4,3.0,4,3,4,11.27,4,Good,809.98,31.944960,266,No,49.574949,21.46538,Low_spent_Large_value_payments,284.629162,Good,Feb,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0
2,3,23,Scientist,19114.12,1824.843333,3,4,3.0,4,3,7,11.27,4,Good,809.98,28.609352,267,No,49.574949,21.46538,Low_spent_Medium_value_payments,331.209863,Good,Mar,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0
3,4,23,Scientist,19114.12,1824.843333,3,4,3.0,4,5,4,6.27,4,Good,809.98,31.377862,268,No,49.574949,21.46538,Low_spent_Small_value_payments,223.451310,Good,Apr,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0
4,5,23,Scientist,19114.12,1824.843333,3,4,3.0,4,6,4,11.27,4,Good,809.98,24.797347,269,No,49.574949,21.46538,High_spent_Medium_value_payments,341.489231,Good,May,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0


In [ ]:
features_to_be_encoded = ["Occupation", "Credit_Mix", "Payment_Behaviour", "Month_Names"]

In [ ]:
data_encoded = pd.concat([pd.get_dummies(data_encoded[features_to_be_encoded], dtype = int), data_encoded], axis = 1)

In [ ]:
data_encoded.head()

,Occupation_Accountant,Occupation_Architect,Occupation_Developer,Occupation_Doctor,Occupation_Engineer,Occupation_Entrepreneur,Occupation_Journalist,Occupation_Lawyer,Occupation_Manager,Occupation_Mechanic,Occupation_Media_Manager,Occupation_Musician,Occupation_Scientist,Occupation_Teacher,Occupation_Writer,Credit_Mix_Bad,Credit_Mix_Good,Credit_Mix_Standard,Payment_Behaviour_High_spent_Large_value_payments,Payment_Behaviour_High_spent_Medium_value_payments,Payment_Behaviour_High_spent_Small_value_payments,Payment_Behaviour_Low_spent_Large_value_payments,Payment_Behaviour_Low_spent_Medium_value_payments,Payment_Behaviour_Low_spent_Small_value_payments,Month_Names_Apr,Month_Names_Aug,Month_Names_Feb,Month_Names_Jan,Month_Names_Jul,Month_Names_Jun,Month_Names_Mar,Month_Names_May,Month,Age,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,Num_Credit_Inquiries,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score,Month_Names,Auto Loan,Credit-Builder Loan,Debt Consolidation Loan,Home Equity Loan,Mortgage Loan,No Data,Not Specified,Payday Loan,Personal Loan,Student Loan,and Auto Loan,and Credit-Builder Loan,and Debt Consolidation Loan,and Home Equity Loan,and Mortgage Loan,and Not Specified,and Payday Loan,and Personal Loan,and Student Loan
0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,23,Scientist,19114.12,1824.843333,3,4,3.0,4,3,7,11.27,4,Good,809.98,26.822620,265,No,49.574949,21.46538,High_spent_Small_value_payments,312.494089,Good,Jan,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,2,23,Scientist,19114.12,1824.843333,3,4,3.0,4,3,4,11.27,4,Good,809.98,31.944960,266,No,49.574949,21.46538,Low_spent_Large_value_payments,284.629162,Good,Feb,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,3,23,Scientist,19114.12,1824.843333,3,4,3.0,4,3,7,11.27,4,Good,809.98,28.609352,267,No,49.574949,21.46538,Low_spent_Medium_value_payments,331.209863,Good,Mar,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,4,23,Scientist,19114.12,1824.843333,3,4,3.0,4,5,4,6.27,4,Good,809.98,31.377862,268,No,49.574949,21.46538,Low_spent_Small_value_payments,223.451310,Good,Apr,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,5,23,Scientist,19114.12,1824.843333,3,4,3.0,4,6,4,11.27,4,Good,809.98,24.797347,269,No,49.574949,21.46538,High_spent_Medium_value_payments,341.489231,Good,May,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0


Removing categorical columns

In [ ]:
data_encoded = data_encoded.drop(columns = features_to_be_encoded)

In [ ]:
data_encoded.head()

,Occupation_Accountant,Occupation_Architect,Occupation_Developer,Occupation_Doctor,Occupation_Engineer,Occupation_Entrepreneur,Occupation_Journalist,Occupation_Lawyer,Occupation_Manager,Occupation_Mechanic,Occupation_Media_Manager,Occupation_Musician,Occupation_Scientist,Occupation_Teacher,Occupation_Writer,Credit_Mix_Bad,Credit_Mix_Good,Credit_Mix_Standard,Payment_Behaviour_High_spent_Large_value_payments,Payment_Behaviour_High_spent_Medium_value_payments,Payment_Behaviour_High_spent_Small_value_payments,Payment_Behaviour_Low_spent_Large_value_payments,Payment_Behaviour_Low_spent_Medium_value_payments,Payment_Behaviour_Low_spent_Small_value_payments,Month_Names_Apr,Month_Names_Aug,Month_Names_Feb,Month_Names_Jan,Month_Names_Jul,Month_Names_Jun,Month_Names_Mar,Month_Names_May,Month,Age,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,Num_Credit_Inquiries,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Monthly_Balance,Credit_Score,Auto Loan,Credit-Builder Loan,Debt Consolidation Loan,Home Equity Loan,Mortgage Loan,No Data,Not Specified,Payday Loan,Personal Loan,Student Loan,and Auto Loan,and Credit-Builder Loan,and Debt Consolidation Loan,and Home Equity Loan,and Mortgage Loan,and Not Specified,and Payday Loan,and Personal Loan,and Student Loan
0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,23,19114.12,1824.843333,3,4,3.0,4,3,7,11.27,4,809.98,26.822620,265,No,49.574949,21.46538,312.494089,Good,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,2,23,19114.12,1824.843333,3,4,3.0,4,3,4,11.27,4,809.98,31.944960,266,No,49.574949,21.46538,284.629162,Good,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,3,23,19114.12,1824.843333,3,4,3.0,4,3,7,11.27,4,809.98,28.609352,267,No,49.574949,21.46538,331.209863,Good,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,4,23,19114.12,1824.843333,3,4,3.0,4,5,4,6.27,4,809.98,31.377862,268,No,49.574949,21.46538,223.451310,Good,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,5,23,19114.12,1824.843333,3,4,3.0,4,6,4,11.27,4,809.98,24.797347,269,No,49.574949,21.46538,341.489231,Good,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0


In [ ]:
data_encoded.shape

(100000, 71)

In [ ]:
data_encoded.columns

Index(['Occupation_Accountant', 'Occupation_Architect', 'Occupation_Developer',
       'Occupation_Doctor', 'Occupation_Engineer', 'Occupation_Entrepreneur',
       'Occupation_Journalist', 'Occupation_Lawyer', 'Occupation_Manager',
       'Occupation_Mechanic', 'Occupation_Media_Manager',
       'Occupation_Musician', 'Occupation_Scientist', 'Occupation_Teacher',
       'Occupation_Writer', 'Credit_Mix_Bad', 'Credit_Mix_Good',
       'Credit_Mix_Standard',
       'Payment_Behaviour_High_spent_Large_value_payments',
       'Payment_Behaviour_High_spent_Medium_value_payments',
       'Payment_Behaviour_High_spent_Small_value_payments',
       'Payment_Behaviour_Low_spent_Large_value_payments',
       'Payment_Behaviour_Low_spent_Medium_value_payments',
       'Payment_Behaviour_Low_spent_Small_value_payments', 'Month_Names_Apr',
       'Month_Names_Aug', 'Month_Names_Feb', 'Month_Names_Jan',
       'Month_Names_Jul', 'Month_Names_Jun', 'Month_Names_Mar',
       'Month_Names_May', 'Month

In [ ]:
"Occupation" in data_encoded.columns

False

In [ ]:
"Payment_of_Min_Amount" in data_encoded.columns

True

### 10. Encoding Target Column: Credit Score

In [ ]:
data_encoded["Credit_Score"].head()

,Credit_Score
0,Good
1,Good
2,Good
3,Good
4,Good


In [ ]:
label_encoder = LabelEncoder()

In [ ]:
data_encoded["Credit_Score_Encoded"] = label_encoder.fit_transform(data_encoded["Credit_Score"])

In [ ]:
data_encoded["Credit_Score_Encoded"].head()

,Credit_Score_Encoded
0,0
1,0
2,0
3,0
4,0


In [ ]:
data_encoded["Credit_Score_Encoded"].value_counts()

,count
Credit_Score_Encoded,
2,53174
1,28998
0,17828


In [ ]:
data["Credit_Score"].value_counts()

,count
Credit_Score,
Standard,53174
Poor,28998
Good,17828


Standard : 2  |  Poor : 1  |  Good : 0

In [ ]:
data_encoded = data_encoded.drop(columns = ["Credit_Score"])

In [ ]:
data_encoded.head()

,Occupation_Accountant,Occupation_Architect,Occupation_Developer,Occupation_Doctor,Occupation_Engineer,Occupation_Entrepreneur,Occupation_Journalist,Occupation_Lawyer,Occupation_Manager,Occupation_Mechanic,Occupation_Media_Manager,Occupation_Musician,Occupation_Scientist,Occupation_Teacher,Occupation_Writer,Credit_Mix_Bad,Credit_Mix_Good,Credit_Mix_Standard,Payment_Behaviour_High_spent_Large_value_payments,Payment_Behaviour_High_spent_Medium_value_payments,Payment_Behaviour_High_spent_Small_value_payments,Payment_Behaviour_Low_spent_Large_value_payments,Payment_Behaviour_Low_spent_Medium_value_payments,Payment_Behaviour_Low_spent_Small_value_payments,Month_Names_Apr,Month_Names_Aug,Month_Names_Feb,Month_Names_Jan,Month_Names_Jul,Month_Names_Jun,Month_Names_Mar,Month_Names_May,Month,Age,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,Num_Credit_Inquiries,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Monthly_Balance,Auto Loan,Credit-Builder Loan,Debt Consolidation Loan,Home Equity Loan,Mortgage Loan,No Data,Not Specified,Payday Loan,Personal Loan,Student Loan,and Auto Loan,and Credit-Builder Loan,and Debt Consolidation Loan,and Home Equity Loan,and Mortgage Loan,and Not Specified,and Payday Loan,and Personal Loan,and Student Loan,Credit_Score_Encoded
0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,23,19114.12,1824.843333,3,4,3.0,4,3,7,11.27,4,809.98,26.822620,265,No,49.574949,21.46538,312.494089,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,2,23,19114.12,1824.843333,3,4,3.0,4,3,4,11.27,4,809.98,31.944960,266,No,49.574949,21.46538,284.629162,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,3,23,19114.12,1824.843333,3,4,3.0,4,3,7,11.27,4,809.98,28.609352,267,No,49.574949,21.46538,331.209863,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,4,23,19114.12,1824.843333,3,4,3.0,4,5,4,6.27,4,809.98,31.377862,268,No,49.574949,21.46538,223.451310,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,5,23,19114.12,1824.843333,3,4,3.0,4,6,4,11.27,4,809.98,24.797347,269,No,49.574949,21.46538,341.489231,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0


In [ ]:
"Credit_Score" in data_encoded.columns

False

### 11. Payment_of_Min_Amount

There are three categories : Yes, No and NM

Yes and No we can understand, but we don't know about NM

Using DecisionTreeClassifier to predict most probable value for NM among Yes and No

In [ ]:
data["Payment_of_Min_Amount"].value_counts()

,count
Payment_of_Min_Amount,
Yes,52326
No,35667
NM,12007


Separate known and unknown rows

In [ ]:
known = data_encoded[data_encoded["Payment_of_Min_Amount"] != "NM"]

In [ ]:
unknown = data_encoded[data_encoded["Payment_of_Min_Amount"] == "NM"]

Prepare X and y

In [ ]:
X_train = known.drop(columns = ["Credit_Score_Encoded", "Payment_of_Min_Amount"], axis = 1)

In [ ]:
y_train = known["Payment_of_Min_Amount"]

In [ ]:
X_test = unknown.drop(columns = ["Credit_Score_Encoded", "Payment_of_Min_Amount"])

Train a classifier

In [ ]:
model = DecisionTreeClassifier(random_state = 42)

In [ ]:
model.fit(X_train, y_train)

DecisionTreeClassifier(random_state=42)

Predict NM values

In [ ]:
predictions = model.predict(X_test)

In [ ]:
predictions

array(['No', 'No', 'Yes', ..., 'No', 'No', 'No'], dtype=object)

Replace NM values

In [ ]:
data_encoded.loc[data_encoded["Payment_of_Min_Amount"] == "NM", "Payment_of_Min_Amount"] = predictions

Check result

In [ ]:
data_encoded["Payment_of_Min_Amount"].value_counts()

,count
Payment_of_Min_Amount,
Yes,59422
No,40578


In [ ]:
data["Payment_of_Min_Amount"].value_counts()

,count
Payment_of_Min_Amount,
Yes,52326
No,35667
NM,12007


### 12. Encoding Payment_of_Min_Amount column

In [ ]:
data_encoded = pd.concat([pd.get_dummies(data_encoded["Payment_of_Min_Amount"], dtype = int).rename(columns={'No': 'Payment_of_Min_Amount_No', 'Yes': 'Payment_of_Min_Amount_Yes'}), data_encoded], axis = 1).drop(columns = ["Payment_of_Min_Amount"], axis = 1)

In [ ]:
data_encoded.head()

,Payment_of_Min_Amount_No,Payment_of_Min_Amount_Yes,Occupation_Accountant,Occupation_Architect,Occupation_Developer,Occupation_Doctor,Occupation_Engineer,Occupation_Entrepreneur,Occupation_Journalist,Occupation_Lawyer,Occupation_Manager,Occupation_Mechanic,Occupation_Media_Manager,Occupation_Musician,Occupation_Scientist,Occupation_Teacher,Occupation_Writer,Credit_Mix_Bad,Credit_Mix_Good,Credit_Mix_Standard,Payment_Behaviour_High_spent_Large_value_payments,Payment_Behaviour_High_spent_Medium_value_payments,Payment_Behaviour_High_spent_Small_value_payments,Payment_Behaviour_Low_spent_Large_value_payments,Payment_Behaviour_Low_spent_Medium_value_payments,Payment_Behaviour_Low_spent_Small_value_payments,Month_Names_Apr,Month_Names_Aug,Month_Names_Feb,Month_Names_Jan,Month_Names_Jul,Month_Names_Jun,Month_Names_Mar,Month_Names_May,Month,Age,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,Num_Credit_Inquiries,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Total_EMI_per_month,Amount_invested_monthly,Monthly_Balance,Auto Loan,Credit-Builder Loan,Debt Consolidation Loan,Home Equity Loan,Mortgage Loan,No Data,Not Specified,Payday Loan,Personal Loan,Student Loan,and Auto Loan,and Credit-Builder Loan,and Debt Consolidation Loan,and Home Equity Loan,and Mortgage Loan,and Not Specified,and Payday Loan,and Personal Loan,and Student Loan,Credit_Score_Encoded
0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,23,19114.12,1824.843333,3,4,3.0,4,3,7,11.27,4,809.98,26.822620,265,49.574949,21.46538,312.494089,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0
1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,2,23,19114.12,1824.843333,3,4,3.0,4,3,4,11.27,4,809.98,31.944960,266,49.574949,21.46538,284.629162,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,3,23,19114.12,1824.843333,3,4,3.0,4,3,7,11.27,4,809.98,28.609352,267,49.574949,21.46538,331.209863,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,4,23,19114.12,1824.843333,3,4,3.0,4,5,4,6.27,4,809.98,31.377862,268,49.574949,21.46538,223.451310,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0
4,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,5,23,19114.12,1824.843333,3,4,3.0,4,6,4,11.27,4,809.98,24.797347,269,49.574949,21.46538,341.489231,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0


In [ ]:
data_encoded.shape

(100000, 72)

In [ ]:
data_encoded["Credit_Score_Encoded"].value_counts()

,count
Credit_Score_Encoded,
2,53174
1,28998
0,17828


## ***4. ML Model Implementation***

### **Decision Tree Classifier**

In [ ]:
data_encoded.head()

,Payment_of_Min_Amount_No,Payment_of_Min_Amount_Yes,Occupation_Accountant,Occupation_Architect,Occupation_Developer,Occupation_Doctor,Occupation_Engineer,Occupation_Entrepreneur,Occupation_Journalist,Occupation_Lawyer,Occupation_Manager,Occupation_Mechanic,Occupation_Media_Manager,Occupation_Musician,Occupation_Scientist,Occupation_Teacher,Occupation_Writer,Credit_Mix_Bad,Credit_Mix_Good,Credit_Mix_Standard,Payment_Behaviour_High_spent_Large_value_payments,Payment_Behaviour_High_spent_Medium_value_payments,Payment_Behaviour_High_spent_Small_value_payments,Payment_Behaviour_Low_spent_Large_value_payments,Payment_Behaviour_Low_spent_Medium_value_payments,Payment_Behaviour_Low_spent_Small_value_payments,Month_Names_Apr,Month_Names_Aug,Month_Names_Feb,Month_Names_Jan,Month_Names_Jul,Month_Names_Jun,Month_Names_Mar,Month_Names_May,Month,Age,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,Num_Credit_Inquiries,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Total_EMI_per_month,Amount_invested_monthly,Monthly_Balance,Auto Loan,Credit-Builder Loan,Debt Consolidation Loan,Home Equity Loan,Mortgage Loan,No Data,Not Specified,Payday Loan,Personal Loan,Student Loan,and Auto Loan,and Credit-Builder Loan,and Debt Consolidation Loan,and Home Equity Loan,and Mortgage Loan,and Not Specified,and Payday Loan,and Personal Loan,and Student Loan,Credit_Score_Encoded
0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,23,19114.12,1824.843333,3,4,3.0,4,3,7,11.27,4,809.98,26.822620,265,49.574949,21.46538,312.494089,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0
1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,2,23,19114.12,1824.843333,3,4,3.0,4,3,4,11.27,4,809.98,31.944960,266,49.574949,21.46538,284.629162,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,3,23,19114.12,1824.843333,3,4,3.0,4,3,7,11.27,4,809.98,28.609352,267,49.574949,21.46538,331.209863,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,4,23,19114.12,1824.843333,3,4,3.0,4,5,4,6.27,4,809.98,31.377862,268,49.574949,21.46538,223.451310,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0
4,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,5,23,19114.12,1824.843333,3,4,3.0,4,6,4,11.27,4,809.98,24.797347,269,49.574949,21.46538,341.489231,1,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0


Separate X and y

In [ ]:
X = data_encoded.drop(columns = ["Credit_Score_Encoded", "Month"], axis = 1)

In [ ]:
y = data_encoded["Credit_Score_Encoded"]

Split the dataset

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Create model

In [ ]:
model = DecisionTreeClassifier(random_state = 42)

Train the model

In [ ]:
model.fit(X_train, y_train)

DecisionTreeClassifier(random_state=42)

In [ ]:
y_pred = model.predict(X_test)

Evaluating the model

In [ ]:
print("Accuracy:- ", accuracy_score(y_test, y_pred))

Accuracy:-  0.7386


In [ ]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.68      0.68      0.68      3527
           1       0.73      0.72      0.72      5874
           2       0.76      0.77      0.77     10599

    accuracy                           0.74     20000
   macro avg       0.72      0.72      0.72     20000
weighted avg       0.74      0.74      0.74     20000



0 : Good  |  1 : Poor  |  2 : Standard

In [ ]:
data_encoded["Credit_Score_Encoded"].value_counts()

,count
Credit_Score_Encoded,
2,53174
1,28998
0,17828


In [ ]:
model.classes_

array([0, 1, 2])

Overfitting is there.

In [ ]:
model.score(X_train, y_train)

1.0

In [ ]:
model.score(X_test, y_test)

0.7386

#### Hyperparameter Tuning

##### Finding best max_depth value

In [ ]:
max_depth = [5, 10, 15, 20, None]

In [ ]:
results = {}

In [ ]:
for i in max_depth:
  model = DecisionTreeClassifier(random_state=42, max_depth = i)
  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)
  results[i] = [accuracy_score(y_test, y_pred), classification_report(y_test, y_pred)]

In [ ]:
for depth in max_depth:
  print(f"--------------- max_depth = {depth} ---------------")
  print("Accuracy:- ", results[depth][0])
  print()
  print(results[depth][1])
  print()
  print("************************************************")

--------------- max_depth = 5 ---------------
Accuracy:-  0.70685

              precision    recall  f1-score   support

           0       0.58      0.67      0.62      3527
           1       0.72      0.65      0.68      5874
           2       0.75      0.75      0.75     10599

    accuracy                           0.71     20000
   macro avg       0.68      0.69      0.69     20000
weighted avg       0.71      0.71      0.71     20000


************************************************
--------------- max_depth = 10 ---------------
Accuracy:-  0.72105

              precision    recall  f1-score   support

           0       0.61      0.64      0.62      3527
           1       0.75      0.67      0.71      5874
           2       0.74      0.78      0.76     10599

    accuracy                           0.72     20000
   macro avg       0.70      0.69      0.70     20000
weighted avg       0.72      0.72      0.72     20000


************************************************
---

max_depth = 20

##### Finding best value for min_samples_split

In [ ]:
min_samples_split = [2, 5, 10, 15, 20]

In [ ]:
results_2 = {}

In [ ]:
for i in min_samples_split:
  model = DecisionTreeClassifier(random_state=42, max_depth = 20, min_samples_split = i)
  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)
  results_2[i] = [accuracy_score(y_test, y_pred), classification_report(y_test, y_pred)]

In [ ]:
for mss in min_samples_split:
  print(f"--------------- min_samples_split = {mss} ---------------")
  print("Accuracy:- ", results_2[mss][0])
  print()
  print(results_2[mss][1])
  print()
  print("************************************************")

--------------- min_samples_split = 2 ---------------
Accuracy:-  0.73575

              precision    recall  f1-score   support

           0       0.65      0.67      0.66      3527
           1       0.74      0.72      0.73      5874
           2       0.76      0.77      0.77     10599

    accuracy                           0.74     20000
   macro avg       0.72      0.72      0.72     20000
weighted avg       0.74      0.74      0.74     20000


************************************************
--------------- min_samples_split = 5 ---------------
Accuracy:-  0.7377

              precision    recall  f1-score   support

           0       0.65      0.67      0.66      3527
           1       0.74      0.72      0.73      5874
           2       0.77      0.77      0.77     10599

    accuracy                           0.74     20000
   macro avg       0.72      0.72      0.72     20000
weighted avg       0.74      0.74      0.74     20000


**************************************

min_samples_split = 10

##### Finding best value for min_samples_leaf

In [ ]:
min_samples_leaf = [1, 2, 5, 10]

In [ ]:
results_3 = {}

In [ ]:
for i in min_samples_leaf:
  model = DecisionTreeClassifier(random_state=42, max_depth = 20, min_samples_split = 10, min_samples_leaf = i)
  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)
  results_3[i] = [accuracy_score(y_test, y_pred), classification_report(y_test, y_pred)]

In [ ]:
for mss in min_samples_leaf:
  print(f"--------------- min_samples_leaf = {mss} ---------------")
  print("Accuracy:- ", results_3[mss][0])
  print()
  print(results_3[mss][1])
  print()
  print("************************************************")

--------------- min_samples_leaf = 1 ---------------
Accuracy:-  0.73815

              precision    recall  f1-score   support

           0       0.64      0.68      0.66      3527
           1       0.74      0.73      0.73      5874
           2       0.77      0.76      0.77     10599

    accuracy                           0.74     20000
   macro avg       0.72      0.72      0.72     20000
weighted avg       0.74      0.74      0.74     20000


************************************************
--------------- min_samples_leaf = 2 ---------------
Accuracy:-  0.7406

              precision    recall  f1-score   support

           0       0.64      0.68      0.66      3527
           1       0.74      0.74      0.74      5874
           2       0.77      0.76      0.77     10599

    accuracy                           0.74     20000
   macro avg       0.72      0.73      0.72     20000
weighted avg       0.74      0.74      0.74     20000


****************************************

min_samples_leaf = 5

##### Finding best value for criterion

In [ ]:
criterion = ["gini", "entropy"]

In [ ]:
results_4 = {}

In [ ]:
for i in criterion:
  model = DecisionTreeClassifier(random_state=42, max_depth = 20, min_samples_split = 10, min_samples_leaf = 5, criterion = i)
  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)
  results_4[i] = [accuracy_score(y_test, y_pred), classification_report(y_test, y_pred)]

In [ ]:
for c in criterion:
  print(f"--------------- criterion = {c} ---------------")
  print("Accuracy:- ", results_4[c][0])
  print()
  print(results_4[c][1])
  print()
  print("************************************************")

--------------- criterion = gini ---------------
Accuracy:-  0.7388

              precision    recall  f1-score   support

           0       0.64      0.68      0.66      3527
           1       0.74      0.73      0.74      5874
           2       0.77      0.77      0.77     10599

    accuracy                           0.74     20000
   macro avg       0.72      0.72      0.72     20000
weighted avg       0.74      0.74      0.74     20000


************************************************
--------------- criterion = entropy ---------------
Accuracy:-  0.74095

              precision    recall  f1-score   support

           0       0.68      0.72      0.70      3527
           1       0.72      0.74      0.73      5874
           2       0.78      0.75      0.76     10599

    accuracy                           0.74     20000
   macro avg       0.72      0.74      0.73     20000
weighted avg       0.74      0.74      0.74     20000


*********************************************

criterion = entropy

##### Finding best value for max_features

In [ ]:
max_features = [None, "sqrt", "log2"]

In [ ]:
results_5 = {}

In [ ]:
for i in max_features:
  model = DecisionTreeClassifier(random_state=42, max_depth = 20, min_samples_split = 10, min_samples_leaf = 5, criterion = "entropy", max_features = i)
  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)
  results_5[i] = [accuracy_score(y_test, y_pred), classification_report(y_test, y_pred)]

In [ ]:
for m in max_features:
  print(f"--------------- max_features = {m} ---------------")
  print("Accuracy:- ", results_5[m][0])
  print()
  print(results_5[m][1])
  print()
  print("************************************************")

--------------- max_features = None ---------------
Accuracy:-  0.74095

              precision    recall  f1-score   support

           0       0.68      0.72      0.70      3527
           1       0.72      0.74      0.73      5874
           2       0.78      0.75      0.76     10599

    accuracy                           0.74     20000
   macro avg       0.72      0.74      0.73     20000
weighted avg       0.74      0.74      0.74     20000


************************************************
--------------- max_features = sqrt ---------------
Accuracy:-  0.7196

              precision    recall  f1-score   support

           0       0.62      0.66      0.64      3527
           1       0.71      0.73      0.72      5874
           2       0.77      0.73      0.75     10599

    accuracy                           0.72     20000
   macro avg       0.70      0.71      0.70     20000
weighted avg       0.72      0.72      0.72     20000


******************************************

max_features = None

##### Final model with best features I got in Hyperparameter Tuning.

In [ ]:
DTC_final_model = DecisionTreeClassifier(random_state=42, max_depth = 20, min_samples_split = 10, min_samples_leaf = 5, criterion = "entropy", max_features = None)

DTC_final_model.fit(X_train, y_train)

DecisionTreeClassifier(criterion='entropy', max_depth=20, min_samples_leaf=5,
                       min_samples_split=10, random_state=42)

In [ ]:
y_pred = DTC_final_model.predict(X_test)

In [ ]:
print(accuracy_score(y_test, y_pred))

0.74095


In [ ]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.68      0.72      0.70      3527
           1       0.72      0.74      0.73      5874
           2       0.78      0.75      0.76     10599

    accuracy                           0.74     20000
   macro avg       0.72      0.74      0.73     20000
weighted avg       0.74      0.74      0.74     20000



#### Using GridSearchCV to find the best hyperparameters and best model with scoring = "accuracy"

In [ ]:
params = {
    "max_depth": [5, 10, 15, 20],
    "min_samples_split": [2, 10, 20],
    "min_samples_leaf": [1, 5, 10],
    "criterion": ["gini", "entropy"]
}

In [ ]:
grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid=params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

In [ ]:
grid.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': [5, 10, 15, 20],
                         'min_samples_leaf': [1, 5, 10],
                         'min_samples_split': [2, 10, 20]},
             scoring='accuracy')

In [ ]:
grid_model = grid.best_estimator_

y_pred_grid = grid_model.predict(X_test)

In [ ]:
accuracy_grid = accuracy_score(y_test, y_pred_grid)

print(accuracy_grid)

0.75215


In [ ]:
class_report_grid = classification_report(y_test, y_pred)

print(class_report_grid)

              precision    recall  f1-score   support

           0       0.67      0.73      0.70      3527
           1       0.74      0.75      0.74      5874
           2       0.79      0.76      0.77     10599

    accuracy                           0.75     20000
   macro avg       0.73      0.74      0.74     20000
weighted avg       0.75      0.75      0.75     20000



In [ ]:
print(grid.best_params_)

{'criterion': 'entropy', 'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 2}


#### Using GridSearchCV to find the best hyperparameters and best model with scoring = "balanced_accuracy"

In [ ]:
grid_2 = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid=params,
    cv=5,
    scoring="balanced_accuracy",
    n_jobs=-1
)

In [ ]:
grid_2.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': [5, 10, 15, 20],
                         'min_samples_leaf': [1, 5, 10],
                         'min_samples_split': [2, 10, 20]},
             scoring='balanced_accuracy')

In [ ]:
grid_2_model = grid_2.best_estimator_

y_pred_grid_2 = grid_2_model.predict(X_test)

In [ ]:
accuracy_grid_2 = accuracy_score(y_test, y_pred_grid)

print(accuracy_grid_2)

0.75215


In [ ]:
class_report_grid_2 = classification_report(y_test, y_pred)

print(class_report_grid_2)

              precision    recall  f1-score   support

           0       0.67      0.73      0.70      3527
           1       0.74      0.75      0.74      5874
           2       0.79      0.76      0.77     10599

    accuracy                           0.75     20000
   macro avg       0.73      0.74      0.74     20000
weighted avg       0.75      0.75      0.75     20000



In [ ]:
print(grid_2.best_params_)

{'criterion': 'entropy', 'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 10}


#### Using GridSearchCV to find the best hyperparameters and best model with scoring = "recall_macro"

In [ ]:
grid_3 = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid=params,
    cv=5,
    scoring="recall_macro",
    n_jobs=-1
)

In [ ]:
grid_3.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': [5, 10, 15, 20],
                         'min_samples_leaf': [1, 5, 10],
                         'min_samples_split': [2, 10, 20]},
             scoring='recall_macro')

In [ ]:
grid_3_model = grid_3.best_estimator_

y_pred_grid_3 = grid_3_model.predict(X_test)

In [ ]:
accuracy_grid_3 = accuracy_score(y_test, y_pred_grid)

print(accuracy_grid_3)

0.75215


In [ ]:
class_report_grid_3 = classification_report(y_test, y_pred)

print(class_report_grid_3)

              precision    recall  f1-score   support

           0       0.67      0.73      0.70      3527
           1       0.74      0.75      0.74      5874
           2       0.79      0.76      0.77     10599

    accuracy                           0.75     20000
   macro avg       0.73      0.74      0.74     20000
weighted avg       0.75      0.75      0.75     20000



In [ ]:
print(grid_3.best_params_)

{'criterion': 'entropy', 'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 10}


### Almost all results are same.

### **Random Forest Classifier**

##### Baseline model : No Hyperparameter Tuning

In [ ]:
model_1 = RandomForestClassifier(random_state=42, n_jobs = -1)

In [ ]:
model_1.fit(X_train, y_train)

RandomForestClassifier(n_jobs=-1, random_state=42)

In [ ]:
y_pred = model_1.predict(X_test)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report")
print(classification_report(y_test, y_pred))

Accuracy: 0.82115

Confusion Matrix
[[2716    8  803]
 [  21 4988  865]
 [ 684 1196 8719]]

Classification Report
              precision    recall  f1-score   support

           0       0.79      0.77      0.78      3527
           1       0.81      0.85      0.83      5874
           2       0.84      0.82      0.83     10599

    accuracy                           0.82     20000
   macro avg       0.81      0.81      0.81     20000
weighted avg       0.82      0.82      0.82     20000



In [ ]:
model_1.score(X_train, y_train)

1.0

In [ ]:
model_1.score(X_test, y_test)

0.82115

#### Training Model with the best hyperparameters which I have got with Decision Tree Classifier

##### 1

In [ ]:
model_2 = RandomForestClassifier(
    n_estimators = 200,
    max_depth = 20,
    min_samples_split = 10,
    min_samples_leaf = 2,
    max_features = None,
    random_state=42,
    n_jobs = -1,
)

In [ ]:
model_2.fit(X_train, y_train)

RandomForestClassifier(max_depth=20, max_features=None, min_samples_leaf=2,
                       min_samples_split=10, n_estimators=200, n_jobs=-1,
                       random_state=42)

In [ ]:
y_pred = model_2.predict(X_test)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report")
print(classification_report(y_test, y_pred))

Accuracy: 0.80845

Confusion Matrix
[[2778   29  720]
 [ 176 4774  924]
 [ 791 1191 8617]]

Classification Report
              precision    recall  f1-score   support

           0       0.74      0.79      0.76      3527
           1       0.80      0.81      0.80      5874
           2       0.84      0.81      0.83     10599

    accuracy                           0.81     20000
   macro avg       0.79      0.80      0.80     20000
weighted avg       0.81      0.81      0.81     20000



##### 2

In [ ]:
model_3 = RandomForestClassifier(
    n_estimators = 200,
    max_depth = 20,
    min_samples_split = 10,
    min_samples_leaf = 2,
    max_features = None,
    random_state=42,
    n_jobs = -1,
    criterion = "entropy"
)

In [ ]:
model_3.fit(X_train, y_train)

RandomForestClassifier(criterion='entropy', max_depth=20, max_features=None,
                       min_samples_leaf=2, min_samples_split=10,
                       n_estimators=200, n_jobs=-1, random_state=42)

In [ ]:
y_pred = model_3.predict(X_test)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report")
print(classification_report(y_test, y_pred))

Accuracy: 0.8136

Confusion Matrix
[[2805   21  701]
 [  55 4840  979]
 [ 735 1237 8627]]

Classification Report
              precision    recall  f1-score   support

           0       0.78      0.80      0.79      3527
           1       0.79      0.82      0.81      5874
           2       0.84      0.81      0.83     10599

    accuracy                           0.81     20000
   macro avg       0.80      0.81      0.81     20000
weighted avg       0.81      0.81      0.81     20000



##### 3

In [ ]:
model_4 = RandomForestClassifier(
    n_estimators = 200,
    max_depth = 20,
    min_samples_split = 10,
    min_samples_leaf = 2,
    max_features = "sqrt",
    random_state=42,
    n_jobs = -1,
    criterion = "entropy"
)

In [ ]:
model_4.fit(X_train, y_train)

RandomForestClassifier(criterion='entropy', max_depth=20, min_samples_leaf=2,
                       min_samples_split=10, n_estimators=200, n_jobs=-1,
                       random_state=42)

In [ ]:
y_pred = model_4.predict(X_test)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report")
print(classification_report(y_test, y_pred))

Accuracy: 0.79695

Confusion Matrix
[[2795   27  705]
 [ 193 4546 1135]
 [ 850 1151 8598]]

Classification Report
              precision    recall  f1-score   support

           0       0.73      0.79      0.76      3527
           1       0.79      0.77      0.78      5874
           2       0.82      0.81      0.82     10599

    accuracy                           0.80     20000
   macro avg       0.78      0.79      0.79     20000
weighted avg       0.80      0.80      0.80     20000



### Selecting Baseline model (Random Forest Classifier) as the best model after comparing results

In [ ]:
RFC_final_model = model_1

## ***5.*** ***Selecting Important Features using Different Methods***

### **Using feature_importances_ and Permutation Mean Importances**

In [ ]:
pd.DataFrame({"Features" : RFC_final_model.feature_names_in_, "Importances" : RFC_final_model.feature_importances_}).sort_values(by = "Importances", ascending = False)

,Features,Importances
45,Outstanding_Debt,0.089566
39,Interest_Rate,0.062165
47,Credit_History_Age,0.053701
41,Delay_from_due_date,0.051919
43,Changed_Credit_Limit,0.046606
...,...,...
16,Occupation_Writer,0.002760
8,Occupation_Journalist,0.002739
13,Occupation_Musician,0.002730
10,Occupation_Manager,0.002703


In [ ]:
result = permutation_importance(RFC_final_model, X_test, y_test, n_repeats = 10, random_state = 42, n_jobs = 1)

In [ ]:
per_imp_df = pd.DataFrame({"Feature" : X_test.columns, "Importance" : result.importances_mean})

In [ ]:
per_imp_df.sort_values(by = "Importance", ascending = True).head()

,Feature,Importance
31,Month_Names_Jun,-0.000825
46,Credit_Utilization_Ratio,-0.000815
30,Month_Names_Jul,-0.000430
24,Payment_Behaviour_Low_spent_Medium_value_payments,-0.000420
26,Month_Names_Apr,-0.000365


#### 1.1 Removing 'Month_Names_Jun', 'Credit_Utilization_Ratio', 'Month_Names_Jul', 'Payment_Behaviour_Low_spent_Medium_value_payments', 'Month_Names_Apr'

In [ ]:
first_drop = ['Month_Names_Jun', 'Credit_Utilization_Ratio', 'Month_Names_Jul', 'Payment_Behaviour_Low_spent_Medium_value_payments', 'Month_Names_Apr']

In [ ]:
X2 = X.drop(columns = ['Month_Names_Jun', 'Credit_Utilization_Ratio', 'Month_Names_Jul', 'Payment_Behaviour_Low_spent_Medium_value_payments', 'Month_Names_Apr'], axis = 1)

Splitting X2 and y

In [ ]:
X2_train, X2_test, y_train, y_test = train_test_split(X2, y, test_size=0.2, random_state=42)

Training model

In [ ]:
model_2 = RandomForestClassifier(random_state=42, n_jobs = -1)

In [ ]:
model_2.fit(X2_train, y_train)

RandomForestClassifier(n_jobs=-1, random_state=42)

Making preditions

In [ ]:
y2_pred = model_2.predict(X2_test)

Evaluating model

In [ ]:
print("Accuracy:", accuracy_score(y_test, y2_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y2_pred))

print("\nClassification Report")
print(classification_report(y_test, y2_pred))

Accuracy: 0.8282

Confusion Matrix
[[2803    5  719]
 [  18 5042  814]
 [ 689 1191 8719]]

Classification Report
              precision    recall  f1-score   support

           0       0.80      0.79      0.80      3527
           1       0.81      0.86      0.83      5874
           2       0.85      0.82      0.84     10599

    accuracy                           0.83     20000
   macro avg       0.82      0.83      0.82     20000
weighted avg       0.83      0.83      0.83     20000



Evaluation metrics are improved than the base model (RFC_final_model).

Using permutation importances

In [ ]:
result_2 = permutation_importance(model_2, X2_test, y_test, n_repeats = 10, random_state = 42, n_jobs = 1)

In [ ]:
per_imp_df_2 = pd.DataFrame({"Feature" : X2_test.columns, "Importance" : result_2.importances_mean})

In [ ]:
per_imp_df_2.sort_values(by = "Importance", ascending = True).head()

,Feature,Importance
20,Payment_Behaviour_High_spent_Large_value_payments,-0.000935
17,Credit_Mix_Bad,-0.000690
29,Month_Names_May,-0.000660
62,and Payday Loan,-0.000355
25,Month_Names_Aug,-0.000255


#### 1.2 Removing "Payment_Behaviour_High_spent_Large_value_payments", "Credit_Mix_Bad", "Month_Names_May", "and Payday Loan", "Month_Names_Aug"



In [ ]:
second_drop = ["Payment_Behaviour_High_spent_Large_value_payments", "Credit_Mix_Bad", "Month_Names_May", "and Payday Loan", "Month_Names_Aug"]

In [ ]:
X3 = X2.drop(columns = second_drop)

Splitting X3 and y

In [ ]:
X3_train, X3_test, y_train, y_test = train_test_split(X3, y, test_size=0.2, random_state=42)

Training Model

In [ ]:
model_3 = RandomForestClassifier(random_state=42, n_jobs = -1)

In [ ]:
model_3.fit(X3_train, y_train)

RandomForestClassifier(n_jobs=-1, random_state=42)

Making Predictions

In [ ]:
y3_pred = model_3.predict(X3_test)

Model Evaluation

In [ ]:
print("Accuracy:", accuracy_score(y_test, y3_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y3_pred))

print("\nClassification Report")
print(classification_report(y_test, y3_pred))

Accuracy: 0.83055

Confusion Matrix
[[2821    6  700]
 [  14 5054  806]
 [ 694 1169 8736]]

Classification Report
              precision    recall  f1-score   support

           0       0.80      0.80      0.80      3527
           1       0.81      0.86      0.84      5874
           2       0.85      0.82      0.84     10599

    accuracy                           0.83     20000
   macro avg       0.82      0.83      0.82     20000
weighted avg       0.83      0.83      0.83     20000



Using permutation importances

In [ ]:
result_3 = permutation_importance(model_3, X3_test, y_test, n_repeats = 10, random_state = 42, n_jobs = 1)

In [ ]:
per_imp_df_3 = pd.DataFrame({"Feature" : X3_test.columns, "Importance" : result_3.importances_mean})

In [ ]:
per_imp_df_3.sort_values(by = "Importance", ascending = True).head()

,Feature,Importance
21,Payment_Behaviour_Low_spent_Large_value_payments,-0.000335
15,Occupation_Teacher,-0.000190
7,Occupation_Entrepreneur,-0.000095
20,Payment_Behaviour_High_spent_Small_value_payments,-0.000085
10,Occupation_Manager,0.000080


#### 1.3 Removing "Payment_Behaviour_Low_spent_Large_value_payments", "Occupation_Teacher", "Occupation_Entrepreneur", "Payment_Behaviour_High_spent_Small_value_payments"

In [ ]:
third_drop = ["Payment_Behaviour_Low_spent_Large_value_payments", "Occupation_Teacher", "Occupation_Entrepreneur", "Payment_Behaviour_High_spent_Small_value_payments"]

In [ ]:
X4 = X3.drop(columns = third_drop, axis = 1)

Splitting X4 and y

In [ ]:
X4_train, X4_test, y_train, y_test = train_test_split(X4, y, test_size=0.2, random_state=42)

Training Model

In [ ]:
model_4 = RandomForestClassifier(random_state=42, n_jobs = -1)

In [ ]:
model_4.fit(X4_train, y_train)

RandomForestClassifier(n_jobs=-1, random_state=42)

Making Predictions

In [ ]:
y4_pred = model_4.predict(X4_test)

Model Evaluation

In [ ]:
print("Accuracy:", accuracy_score(y_test, y4_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y4_pred))

print("\nClassification Report")
print(classification_report(y_test, y4_pred))

Accuracy: 0.83225

Confusion Matrix
[[2849    5  673]
 [  16 5038  820]
 [ 698 1143 8758]]

Classification Report
              precision    recall  f1-score   support

           0       0.80      0.81      0.80      3527
           1       0.81      0.86      0.84      5874
           2       0.85      0.83      0.84     10599

    accuracy                           0.83     20000
   macro avg       0.82      0.83      0.83     20000
weighted avg       0.83      0.83      0.83     20000



Using permuation importances

In [ ]:
result_4 = permutation_importance(model_4, X4_test, y_test, n_repeats = 10, random_state = 42, n_jobs = 1)

In [ ]:
per_imp_df_4 = pd.DataFrame({"Feature" : X4_test.columns, "Importance" : result_4.importances_mean})

In [ ]:
per_imp_df_4.sort_values(by = "Importance", ascending = True).head()

,Feature,Importance
41,Home Equity Loan,-0.000275
42,Mortgage Loan,-0.000265
44,Not Specified,-0.000050
7,Occupation_Journalist,0.000035
9,Occupation_Manager,0.000045


#### 1.4 Removing "Home Equity Loan", "Mortgage Loan", "Not Specified"

In [ ]:
fourth_drop = ["Home Equity Loan", "Mortgage Loan", "Not Specified"]

In [ ]:
X5 = X4.drop(columns = fourth_drop, axis = 1)

Splitting X5 and y

In [ ]:
X5_train, X5_test, y_train, y_test = train_test_split(X5, y, test_size=0.2, random_state=42)

Training Model

In [ ]:
model_5 = RandomForestClassifier(random_state=42, n_jobs = -1)

In [ ]:
model_5.fit(X5_train, y_train)

RandomForestClassifier(n_jobs=-1, random_state=42)

Making predictions

In [ ]:
y5_pred = model_5.predict(X5_test)

Model Evaluation

In [ ]:
print("Accuracy:", accuracy_score(y_test, y5_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y5_pred))

print("\nClassification Report")
print(classification_report(y_test, y5_pred))

Accuracy: 0.8309

Confusion Matrix
[[2838    8  681]
 [  16 5049  809]
 [ 706 1162 8731]]

Classification Report
              precision    recall  f1-score   support

           0       0.80      0.80      0.80      3527
           1       0.81      0.86      0.84      5874
           2       0.85      0.82      0.84     10599

    accuracy                           0.83     20000
   macro avg       0.82      0.83      0.82     20000
weighted avg       0.83      0.83      0.83     20000



Using permuation importances

In [ ]:
result_5 = permutation_importance(model_5, X5_test, y_test, n_repeats = 10, random_state = 42, n_jobs = 1)

In [ ]:
per_imp_df_5 = pd.DataFrame({"Feature" : X5_test.columns, "Importance" : result_5.importances_mean})

In [ ]:
per_imp_df_5.sort_values(by = "Importance", ascending = True).head()

,Feature,Importance
9,Occupation_Manager,-0.000200
42,Payday Loan,-0.000165
45,and Auto Loan,-0.000090
51,and Personal Loan,-0.000045
7,Occupation_Journalist,-0.000030


#### 1.5 Removing "Occupation_Manager", "Payday Loan", "and Auto Loan", "and Personal Loan", "Occupation_Journalist"

In [ ]:
fifth_drop = ["Occupation_Manager", "Payday Loan", "and Auto Loan", "and Personal Loan", "Occupation_Journalist"]

In [ ]:
X6 = X5.drop(columns = fifth_drop, axis = 1)

Splitting X6 and y

In [ ]:
X6_train, X6_test, y_train, y_test = train_test_split(X6, y, test_size=0.2, random_state=42)

Training Model

In [ ]:
model_6 = RandomForestClassifier(random_state=42, n_jobs = -1)

In [ ]:
model_6.fit(X6_train, y_train)

RandomForestClassifier(n_jobs=-1, random_state=42)

Making predictions

In [ ]:
y6_pred = model_6.predict(X6_test)

Model evaluation

In [ ]:
print("Accuracy:", accuracy_score(y_test, y6_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y6_pred))

print("\nClassification Report")
print(classification_report(y_test, y6_pred))

Accuracy: 0.8305

Confusion Matrix
[[2817   12  698]
 [  11 5058  805]
 [ 707 1157 8735]]

Classification Report
              precision    recall  f1-score   support

           0       0.80      0.80      0.80      3527
           1       0.81      0.86      0.84      5874
           2       0.85      0.82      0.84     10599

    accuracy                           0.83     20000
   macro avg       0.82      0.83      0.82     20000
weighted avg       0.83      0.83      0.83     20000



Using permutation importances

In [ ]:
result_6 = permutation_importance(model_6, X6_test, y_test, n_repeats = 10, random_state = 42, n_jobs = 1)

In [ ]:
per_imp_df_6 = pd.DataFrame({"Feature" : X6_test.columns, "Importance" : result_6.importances_mean})

In [ ]:
per_imp_df_6.sort_values(by = "Importance", ascending = True).head()

,Feature,Importance
37,Credit-Builder Loan,-0.000380
40,Personal Loan,-0.000325
42,and Credit-Builder Loan,-0.000100
47,and Student Loan,-0.000025
46,and Not Specified,0.000110


#### 1.6 Removing "Credit-Builder Loan", "Personal Loan", "and Credit-Builder Loan", "and Student Loan"

In [ ]:
sixth_drop = ["Credit-Builder Loan", "Personal Loan", "and Credit-Builder Loan", "and Student Loan"]

In [ ]:
X7 = X6.drop(columns = sixth_drop, axis = 1)

Splitting X7 and y

In [ ]:
X7_train, X7_test, y_train, y_test = train_test_split(X7, y, test_size=0.2, random_state=42)

Training Model

In [ ]:
model_7 = RandomForestClassifier(random_state=42, n_jobs = -1)

In [ ]:
model_7.fit(X7_train, y_train)

RandomForestClassifier(n_jobs=-1, random_state=42)

Making Predicitions

In [ ]:
y7_pred = model_7.predict(X7_test)

Model evaluation

In [ ]:
print("Accuracy:", accuracy_score(y_test, y7_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y7_pred))

print("\nClassification Report")
print(classification_report(y_test, y7_pred))

Accuracy: 0.83165

Confusion Matrix
[[2833    7  687]
 [  13 5056  805]
 [ 684 1171 8744]]

Classification Report
              precision    recall  f1-score   support

           0       0.80      0.80      0.80      3527
           1       0.81      0.86      0.84      5874
           2       0.85      0.82      0.84     10599

    accuracy                           0.83     20000
   macro avg       0.82      0.83      0.83     20000
weighted avg       0.83      0.83      0.83     20000



Using permutation importances

In [ ]:
result_7 = permutation_importance(model_7, X7_test, y_test, n_repeats = 10, random_state = 42, n_jobs = 1)

In [ ]:
per_imp_df_7 = pd.DataFrame({"Feature" : X7_test.columns, "Importance" : result_7.importances_mean})

In [ ]:
per_imp_df_7.sort_values(by = "Importance", ascending = True).head()

,Feature,Importance
11,Occupation_Scientist,0.000040
6,Occupation_Engineer,0.000210
41,and Home Equity Loan,0.000235
43,and Not Specified,0.000265
42,and Mortgage Loan,0.000460


#### 1.7 Removing "Occupation_Scientist", "Occupation_Engineer", "and Home Equity Loan", "and Not Specified", "and Mortgage Loan"

In [ ]:
seventh_drop = ["Occupation_Scientist", "Occupation_Engineer", "and Home Equity Loan", "and Not Specified", "and Mortgage Loan"]

In [ ]:
X8 = X7.drop(columns = seventh_drop, axis = 1)

In [ ]:
X8.shape

(100000, 39)

Splitting X8 and y

In [ ]:
X8_train, X8_test, y_train, y_test = train_test_split(X8, y, test_size=0.2, random_state=42)

Training model

In [ ]:
model_8 = RandomForestClassifier(random_state=42, n_jobs = -1)

In [ ]:
model_8.fit(X8_train, y_train)

RandomForestClassifier(n_jobs=-1, random_state=42)

Making predictions

In [ ]:
y8_pred = model_8.predict(X8_test)

Model evaluation

In [ ]:
print("Accuracy:", accuracy_score(y_test, y8_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y8_pred))

print("\nClassification Report")
print(classification_report(y_test, y8_pred))

Accuracy: 0.8319

Confusion Matrix
[[2826    7  694]
 [  13 5082  779]
 [ 703 1166 8730]]

Classification Report
              precision    recall  f1-score   support

           0       0.80      0.80      0.80      3527
           1       0.81      0.87      0.84      5874
           2       0.86      0.82      0.84     10599

    accuracy                           0.83     20000
   macro avg       0.82      0.83      0.83     20000
weighted avg       0.83      0.83      0.83     20000



Using permuation importances

In [ ]:
result_8 = permutation_importance(model_8, X8_test, y_test, n_repeats = 10, random_state = 42, n_jobs = 1)

In [ ]:
per_imp_df_8 = pd.DataFrame({"Feature" : X8_test.columns, "Importance" : result_8.importances_mean})

In [ ]:
per_imp_df_8.sort_values(by = "Importance", ascending = True).head()

,Feature,Importance
5,Occupation_Doctor,0.000085
13,Payment_Behaviour_High_spent_Medium_value_paym...,0.000260
38,and Debt Consolidation Loan,0.000340
6,Occupation_Lawyer,0.000445
3,Occupation_Architect,0.000455


#### 1.8 Removing "Occupation_Doctor", "Payment_Behaviour_High_spent_Medium_value_payments", "and Debt Consolidation Loan", "Occupation_Lawyer", "Occupation_Architect"

In [ ]:
eighth_drop = ["Occupation_Doctor", "Payment_Behaviour_High_spent_Medium_value_payments", "and Debt Consolidation Loan", "Occupation_Lawyer", "Occupation_Architect"]

In [ ]:
X9 = X8.drop(columns = eighth_drop, axis = 1)

Splitting X9 and y

In [ ]:
X9_train, X9_test, y_train, y_test = train_test_split(X9, y, test_size=0.2, random_state=42)

Training Model

In [ ]:
model_9 = RandomForestClassifier(random_state=42, n_jobs = -1)

In [ ]:
model_9.fit(X9_train, y_train)

RandomForestClassifier(n_jobs=-1, random_state=42)

Making predictions

In [ ]:
y9_pred = model_9.predict(X9_test)

Model evaluation

In [ ]:
print("Accuracy:", accuracy_score(y_test, y9_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y9_pred))

print("\nClassification Report")
print(classification_report(y_test, y9_pred))

Accuracy: 0.83145

Confusion Matrix
[[2833    9  685]
 [  19 5048  807]
 [ 710 1141 8748]]

Classification Report
              precision    recall  f1-score   support

           0       0.80      0.80      0.80      3527
           1       0.81      0.86      0.84      5874
           2       0.85      0.83      0.84     10599

    accuracy                           0.83     20000
   macro avg       0.82      0.83      0.83     20000
weighted avg       0.83      0.83      0.83     20000



Using permutation importances

In [ ]:
result_9 = permutation_importance(model_9, X9_test, y_test, n_repeats = 10, random_state = 42, n_jobs = 1)

In [ ]:
per_imp_df_9 = pd.DataFrame({"Feature" : X9_test.columns, "Importance" : result_9.importances_mean})

In [ ]:
per_imp_df_9.sort_values(by = "Importance", ascending = True).head()

,Feature,Importance
30,Auto Loan,0.000300
5,Occupation_Media_Manager,0.000370
32,No Data,0.000430
2,Occupation_Accountant,0.000605
6,Occupation_Musician,0.000680


#### 1.9 Removing "Auto Loan", "Occupation_Media_Manager", "No_Data", "Occupation_Accountant", "Occupation_Musician"

In [ ]:
ninth_drop = ["Auto Loan", "Occupation_Media_Manager", "No Data", "Occupation_Accountant", "Occupation_Musician"]

In [ ]:
X10 = X9.drop(columns = ninth_drop, axis = 1)

Splitting X10 and y

In [ ]:
X10_train, X10_test, y_train, y_test = train_test_split(X10, y, test_size=0.2, random_state=42)

Training model

In [ ]:
model_10 = RandomForestClassifier(random_state=42, n_jobs = -1)

In [ ]:
model_10.fit(X10_train, y_train)

RandomForestClassifier(n_jobs=-1, random_state=42)

Making predictions

In [ ]:
y10_pred = model_10.predict(X10_test)

Model evaluation

In [ ]:
print("Accuracy:", accuracy_score(y_test, y10_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y10_pred))

print("\nClassification Report")
print(classification_report(y_test, y10_pred))

Accuracy: 0.82995

Confusion Matrix
[[2808   12  707]
 [  16 5049  809]
 [ 714 1143 8742]]

Classification Report
              precision    recall  f1-score   support

           0       0.79      0.80      0.79      3527
           1       0.81      0.86      0.84      5874
           2       0.85      0.82      0.84     10599

    accuracy                           0.83     20000
   macro avg       0.82      0.83      0.82     20000
weighted avg       0.83      0.83      0.83     20000



Using permutation importances

In [ ]:
result_10 = permutation_importance(model_10, X10_test, y_test, n_repeats = 10, random_state = 42, n_jobs = 1)

In [ ]:
per_imp_df_10 = pd.DataFrame({"Feature" : X10_test.columns, "Importance" : result_10.importances_mean})

In [ ]:
per_imp_df_10.sort_values(by = "Importance", ascending = True).head()

,Feature,Importance
4,Occupation_Writer,0.000365
2,Occupation_Developer,0.000365
3,Occupation_Mechanic,0.000575
27,Debt Consolidation Loan,0.000720
28,Student Loan,0.001240


#### 1.10 Removing "Occupation_Writer", "Occupation_Developer", "Occupation_Mechanic", "Debt Consolidation Loan", "Student Loan"

In [ ]:
tenth_drop = ["Occupation_Writer", "Occupation_Developer", "Occupation_Mechanic", "Debt Consolidation Loan", "Student Loan"]

In [ ]:
X11 = X10.drop(columns = tenth_drop, axis = 1)

Splitting X11 and y

In [ ]:
X11_train, X11_test, y_train, y_test = train_test_split(X11, y, test_size=0.2, random_state=42)

Training model

In [ ]:
model_11 = RandomForestClassifier(random_state=42, n_jobs = -1)

In [ ]:
model_11.fit(X11_train, y_train)

RandomForestClassifier(n_jobs=-1, random_state=42)

Making predictions

In [ ]:
y11_pred = model_11.predict(X11_test)

Model evaluation

In [ ]:
print("Accuracy:", accuracy_score(y_test, y11_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y11_pred))

print("\nClassification Report")
print(classification_report(y_test, y11_pred))

Accuracy: 0.83055

Confusion Matrix
[[2826   11  690]
 [  19 5048  807]
 [ 707 1155 8737]]

Classification Report
              precision    recall  f1-score   support

           0       0.80      0.80      0.80      3527
           1       0.81      0.86      0.84      5874
           2       0.85      0.82      0.84     10599

    accuracy                           0.83     20000
   macro avg       0.82      0.83      0.82     20000
weighted avg       0.83      0.83      0.83     20000



Using permuation importances

In [ ]:
result_11 = permutation_importance(model_11, X11_test, y_test, n_repeats = 10, random_state = 42, n_jobs = 1)

In [ ]:
per_imp_df_11 = pd.DataFrame({"Feature" : X11_test.columns, "Importance" : result_11.importances_mean})

In [ ]:
per_imp_df_11.sort_values(by = "Importance", ascending = True).head()

,Feature,Importance
4,Payment_Behaviour_Low_spent_Small_value_payments,0.002765
23,Monthly_Balance,0.002825
6,Month_Names_Jan,0.005930
0,Payment_of_Min_Amount_No,0.006070
5,Month_Names_Feb,0.006215


#### 1.11 Removing "Payment_Behaviour_Low_spent_Small_value_payments", "Monthly_Balance", "Month_Names_Jan", "Payment_of_Min_Amount_No", "Month_Names_Feb"

In [ ]:
eleventh_drop = ["Payment_Behaviour_Low_spent_Small_value_payments", "Monthly_Balance", "Month_Names_Jan", "Payment_of_Min_Amount_No", "Month_Names_Feb"]

In [ ]:
X12 = X11.drop(columns = eleventh_drop, axis = 1)

Splitting X12 and y

In [ ]:
X12_train, X12_test, y_train, y_test = train_test_split(X12, y, test_size=0.2, random_state=42)

Training model

In [ ]:
model_12 = RandomForestClassifier(random_state=42, n_jobs = -1)

In [ ]:
model_12.fit(X12_train, y_train)

RandomForestClassifier(n_jobs=-1, random_state=42)

Making predicitons

In [ ]:
y12_pred = model_12.predict(X12_test)

Model evaluation

In [ ]:
print("Accuracy:", accuracy_score(y_test, y12_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y12_pred))

print("\nClassification Report")
print(classification_report(y_test, y12_pred))

Accuracy: 0.8246

Confusion Matrix
[[2819    9  699]
 [  11 4984  879]
 [ 750 1160 8689]]

Classification Report
              precision    recall  f1-score   support

           0       0.79      0.80      0.79      3527
           1       0.81      0.85      0.83      5874
           2       0.85      0.82      0.83     10599

    accuracy                           0.82     20000
   macro avg       0.81      0.82      0.82     20000
weighted avg       0.83      0.82      0.82     20000



Using permuation importances

In [ ]:
result_12 = permutation_importance(model_12, X12_test, y_test, n_repeats = 10, random_state = 42, n_jobs = 1)

In [ ]:
per_imp_df_12 = pd.DataFrame({"Feature" : X12_test.columns, "Importance" : result_12.importances_mean})

In [ ]:
per_imp_df_12.sort_values(by = "Importance", ascending = True).head()

,Feature,Importance
10,Num_of_Loan,0.001760
4,Age,0.003090
3,Month_Names_Mar,0.005520
6,Monthly_Inhand_Salary,0.006290
5,Annual_Income,0.007265


In [ ]:
per_imp_df_12.sort_values(by = "Importance", ascending = False)

,Feature,Importance
9,Interest_Rate,0.084890
11,Delay_from_due_date,0.068405
15,Outstanding_Debt,0.063970
2,Credit_Mix_Standard,0.052395
13,Changed_Credit_Limit,0.036820
8,Num_Credit_Card,0.035330
16,Credit_History_Age,0.032100
1,Credit_Mix_Good,0.027430
12,Num_of_Delayed_Payment,0.019310
7,Num_Bank_Accounts,0.018835


#### 1.12 Removing "Month_Names_Mar", "Age", "Num_of_Loan"

and adding "Credit_Mix_Bad", "Payment_of_Min_Amount_No"

In [ ]:
X12.columns

Index(['Payment_of_Min_Amount_Yes', 'Credit_Mix_Good', 'Credit_Mix_Standard',
       'Month_Names_Mar', 'Age', 'Annual_Income', 'Monthly_Inhand_Salary',
       'Num_Bank_Accounts', 'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan',
       'Delay_from_due_date', 'Num_of_Delayed_Payment', 'Changed_Credit_Limit',
       'Num_Credit_Inquiries', 'Outstanding_Debt', 'Credit_History_Age',
       'Total_EMI_per_month', 'Amount_invested_monthly'],
      dtype='object')

In [ ]:
twelth_drop = ["Month_Names_Mar", "Age", "Num_of_Loan"]

In [ ]:
X13 = X12.drop(columns = twelth_drop, axis = 1)

In [ ]:
X13["Credit_Mix_Bad"] = data_encoded["Credit_Mix_Bad"]
X13["Payment_of_Min_Amount_No"] = data_encoded["Payment_of_Min_Amount_No"]

In [ ]:
X13.shape

(100000, 18)

Splitting X13 and y

In [ ]:
X13_train, X13_test, y_train, y_test = train_test_split(X13, y, test_size=0.2, random_state=42)

Training model

In [ ]:
model_13 = RandomForestClassifier(random_state=42, n_jobs = -1)

In [ ]:
model_13.fit(X13_train, y_train)

RandomForestClassifier(n_jobs=-1, random_state=42)

Making predictins

In [ ]:
y13_train = model_13.predict(X13_test)

Model evaluation

In [ ]:
print("Accuracy:", accuracy_score(y_test, y13_train))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y13_train))

print("\nClassification Report")
print(classification_report(y_test, y13_train))

Accuracy: 0.81865

Confusion Matrix
[[2777   10  740]
 [  10 4901  963]
 [ 746 1158 8695]]

Classification Report
              precision    recall  f1-score   support

           0       0.79      0.79      0.79      3527
           1       0.81      0.83      0.82      5874
           2       0.84      0.82      0.83     10599

    accuracy                           0.82     20000
   macro avg       0.81      0.81      0.81     20000
weighted avg       0.82      0.82      0.82     20000



Using permuation importances

In [ ]:
result_13 = permutation_importance(model_13, X13_test, y_test, n_repeats = 10, random_state = 42, n_jobs = 1)

In [ ]:
per_imp_df_13 = pd.DataFrame({"Feature" : X13_test.columns, "Importance" : result_13.importances_mean})

In [ ]:
per_imp_df_13.sort_values(by = "Importance", ascending = False)

,Feature,Importance
7,Interest_Rate,0.102035
12,Outstanding_Debt,0.068540
8,Delay_from_due_date,0.068475
10,Changed_Credit_Limit,0.036315
6,Num_Credit_Card,0.035330
13,Credit_History_Age,0.034535
1,Credit_Mix_Good,0.031545
2,Credit_Mix_Standard,0.027805
5,Num_Bank_Accounts,0.019185
9,Num_of_Delayed_Payment,0.015380


#### Using the best hyperparameter on the model obtained at step 1.12

In [ ]:
model_14 = RandomForestClassifier(n_estimators = 200,
    max_depth = 20,
    min_samples_split = 10,
    min_samples_leaf = 2,
    max_features = None,
    random_state=42,
    n_jobs = -1,
    criterion = "entropy")

In [ ]:
model_14.fit(X13_train, y_train)

RandomForestClassifier(criterion='entropy', max_depth=20, max_features=None,
                       min_samples_leaf=2, min_samples_split=10,
                       n_estimators=200, n_jobs=-1, random_state=42)

In [ ]:
y14_train = model_13.predict(X13_test)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y14_train))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y14_train))

print("\nClassification Report")
print(classification_report(y_test, y14_train))

Accuracy: 0.81865

Confusion Matrix
[[2777   10  740]
 [  10 4901  963]
 [ 746 1158 8695]]

Classification Report
              precision    recall  f1-score   support

           0       0.79      0.79      0.79      3527
           1       0.81      0.83      0.82      5874
           2       0.84      0.82      0.83     10599

    accuracy                           0.82     20000
   macro avg       0.81      0.81      0.81     20000
weighted avg       0.82      0.82      0.82     20000



# **Future Work**

## Final set of important features I got using permutation importances method

In [ ]:
X_final = X13

In [ ]:
X_final.columns

Index(['Payment_of_Min_Amount_Yes', 'Credit_Mix_Good', 'Credit_Mix_Standard',
       'Annual_Income', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts',
       'Num_Credit_Card', 'Interest_Rate', 'Delay_from_due_date',
       'Num_of_Delayed_Payment', 'Changed_Credit_Limit',
       'Num_Credit_Inquiries', 'Outstanding_Debt', 'Credit_History_Age',
       'Total_EMI_per_month', 'Amount_invested_monthly', 'Credit_Mix_Bad',
       'Payment_of_Min_Amount_No'],
      dtype='object')

## Final model I got at the 1.12 th step

In [ ]:
final_model = model_13

## Saving final dataset

In [ ]:
final_data = pd.concat([X_final, y], axis = 1)

In [ ]:
final_data.to_csv("credit_score_data_full.csv")

### Saving training dataset

In [ ]:
final_data_train = pd.concat([X13_train, y_train], axis = 1)

In [ ]:
final_data_train.to_csv("credit_score_data_train.csv")

## Saving the model

In [ ]:
joblib.dump(final_model, "credit_score_model.pkl")

['credit_score_model.pkl']

In [ ]:
data["Credit_Score"].value_counts()

,count
Credit_Score,
Standard,53174
Poor,28998
Good,17828


In [ ]:
data_encoded["Credit_Score_Encoded"].value_counts()

,count
Credit_Score_Encoded,
2,53174
1,28998
0,17828


## Future work

* Interpret those 16 features - explain why they matter and what they tell you about the problem.

# **Conclusion**

This project successfully demonstrates the application of machine learning techniques for predicting customer credit scores using financial and credit-related information.

A systematic preprocessing pipeline was developed to transform categorical variables, handle ambiguous values, and prepare the dataset for model training. Both Decision Tree and Random Forest classifiers were evaluated. Although hyperparameter tuning provided only marginal improvements for the Decision Tree model, the Random Forest Classifier achieved substantially better predictive performance, with an accuracy of approximately 82.1%, indicating the effectiveness of ensemble learning for this classification task.

Permutation feature importance was then employed to identify the most influential predictors. Through iterative feature elimination, the original feature set was reduced to 18 important features while maintaining nearly identical model performance (approximately 81.9% accuracy). This demonstrates that many original variables contributed little to prediction accuracy and could be removed without significantly affecting the model's effectiveness.

The final model is both accurate and computationally efficient, making it more suitable for practical deployment in credit risk assessment systems. The identified important features—including annual income, monthly in-hand salary, interest rate, outstanding debt, credit history age, delayed payments, and credit mix—provide valuable insights into the factors most strongly associated with customer credit scores.

As future work, explainable AI techniques such as SHAP, LIME, or partial dependence analysis can be incorporated to improve transparency by explaining how each important feature influences individual credit score predictions. This would enhance model interpretability and increase trust among financial institutions and stakeholders.